In [1]:
"""
chargemaster_extractor.py
--------------------------
Recursively traverses a folder structure organized by year → hospital → files,
extracts ALL rows containing valid CPT or HCPCS codes, and writes three output CSVs.
No target list required — every valid code found is captured.

Expected layout:
    base_dir/
        2013/
            HospitalA/
                chargemaster.xlsx
            HospitalB/
                cdm.csv
        2014/
            ...

Outputs (written next to this script):
    matched_rows.csv        – all matched rows with metadata
    extraction_log.csv      – per-sheet processing summary
    missing_files_or_errors.csv – files that could not be read
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# Root directory containing year folders
BASE_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"

# Where to write the three output files
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"

# ─────────────────────────────────────────────
# TARGET CPT / HCPCS CODES
# ─────────────────────────────────────────────
# Emergency Medicine CPT codes (99281-99285 = ER visits by severity)
# Gynecology CPT codes (common OB/GYN procedures)
# Add or remove codes here as needed for your research.
TARGET_CODES = {
    # ── Emergency Medicine ──────────────────────────────────────────
    "99281", "99282", "99283", "99284", "99285",  # ER visit levels 1-5
    "99291", "99292",                              # Critical care

    # ── Gynecology ──────────────────────────────────────────────────
    "59400", "59409", "59410",                     # Vaginal delivery
    "59510", "59514", "59515",                     # Cesarean delivery
}

# ─────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────

# Column-name patterns that likely contain procedure codes
CODE_COL_PATTERNS = re.compile(
    r"(cpt|hcpcs|procedure\s*code|proc\s*code|cpt[\s/\-]*hcpcs|service\s*code|"
    r"\bcode\b|billing\s*code|revenue\s*code)",
    re.IGNORECASE,
)

# Column-name patterns for description and charge
DESC_COL_PATTERNS = re.compile(
    r"(description|desc|procedure\s*name|service\s*name|item\s*name|narrative)",
    re.IGNORECASE,
)
CHARGE_COL_PATTERNS = re.compile(
    # Match price/amount columns but EXCLUDE "Charge Code" (internal hospital ID)
    # Also catches "June 2024 Prices", "Average Charge", "Gross Charge" etc.
    r"(?i)^(?!charge\s*code)(?=.*(price|amount|rate|average\s*charge|avg\s*charge|"
    r"gross\s*charge|standard\s*charge|billed\s*charge|cdm\s*price|list\s*price|"
    r"\d{4}\s*price))",
    re.IGNORECASE,
)

# CPT codes: 5 digits (optionally followed by a 2-char modifier)
CPT_PATTERN = re.compile(r"^\d{5}([A-Z0-9]{2})?$")

# HCPCS Level II codes: letter + 4 digits
HCPCS_PATTERN = re.compile(r"^[A-Z]\d{4}$")


# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────


def normalize_code(value) -> str:
    """Strip whitespace, uppercase, remove non-alphanumeric characters."""
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())


def detect_code_type(code: str) -> str:
    """Return 'CPT', 'HCPCS', or 'UNKNOWN' based on code format."""
    if CPT_PATTERN.match(code):
        return "CPT"
    if HCPCS_PATTERN.match(code):
        return "HCPCS"
    return "UNKNOWN"


def find_columns(df: pd.DataFrame, pattern: re.Pattern) -> list:
    """Return column names matching a regex pattern."""
    return [c for c in df.columns if pattern.search(str(c))]


# Keywords that strongly indicate a real header row
HEADER_KEYWORDS = re.compile(
    r"(cpt|hcpcs|code|charge|price|amount|description|desc|procedure|service|rate|cost|revenue)",
    re.IGNORECASE,
)

def find_header_row(df_raw: pd.DataFrame, max_scan: int = 15) -> int:
    """
    Find the real header row by scoring each row on two criteria:
    1. KEYWORD score  — how many cells contain header-like words (cpt, charge, description etc.)
    2. BREADTH score  — how many non-empty string cells (width of the header)
    Keyword hits are weighted heavily so title rows like "Common OP Procedures"
    don't outscore a real multi-column header.
    """
    best_row, best_score = 0, -1
    for i in range(min(max_scan, len(df_raw))):
        row = df_raw.iloc[i]
        keyword_hits = sum(
            1 for v in row
            if isinstance(v, str) and HEADER_KEYWORDS.search(v)
        )
        breadth = sum(
            1 for v in row
            if isinstance(v, str) and len(v.strip()) > 1
        )
        # Keyword hits worth 5x more than plain text breadth
        score = keyword_hits * 5 + breadth
        if score > best_score:
            best_score, best_row = score, i
    return best_row


def get_engine(filepath: str) -> str:
    """Return the correct pandas engine based on file extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".xls":
        return "xlrd"
    return "openpyxl"


def read_sheet(filepath: str, sheet_name, engine: str) -> pd.DataFrame:
    """
    Read a single Excel sheet in one pass, auto-detecting the header row.
    Uses xlrd for .xls and openpyxl for .xlsx/.xlsm.
    """
    # Single raw read to find header row and return data
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None,
                        dtype=str, engine=engine)
    header_row = find_header_row(raw)
    # Slice: rows above header become the header, rows below are data
    df = raw.iloc[header_row + 1:].copy()
    df.columns = [str(v).strip() for v in raw.iloc[header_row]]
    df.reset_index(drop=True, inplace=True)
    return df


def read_csv_file(filepath: str) -> pd.DataFrame:
    """Read a CSV, trying common encodings."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(filepath, dtype=str, encoding=enc)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode CSV: {filepath}")


# ─────────────────────────────────────────────
# CORE EXTRACTION LOGIC
# ─────────────────────────────────────────────

def is_valid_code(code: str) -> bool:
    """Return True if code is a valid CPT/HCPCS format AND in our target list."""
    is_valid_format = bool(CPT_PATTERN.match(code) or HCPCS_PATTERN.match(code))
    return is_valid_format and code in TARGET_CODES


def extract_matches(
    df: pd.DataFrame,
    year: str,
    hospital: str,
    filename: str,
    sheet_name: str,
) -> tuple[list[dict], dict]:
    """
    Scan `df` and extract ALL rows that contain a valid CPT or HCPCS code.

    Returns:
        matched_rows  – list of dicts (one per matched row)
        log_entry     – dict summarising what was detected in this sheet
    """
    # Identify candidate code columns by name
    code_cols = find_columns(df, CODE_COL_PATTERNS)
    desc_cols = find_columns(df, DESC_COL_PATTERNS)
    charge_cols = find_columns(df, CHARGE_COL_PATTERNS)

    # If no named code column found, scan ALL columns for target code values
    if not code_cols:
        for col in df.columns:
            sample = df[col].dropna().head(100)
            hits = sample.apply(lambda v: is_valid_code(normalize_code(v))).sum()
            if hits >= 1:  # even 1 target code hit is enough
                code_cols.append(col)

    matched_rows = []

    for code_col in code_cols:
        for _, row in df.iterrows():
            raw_val = row.get(code_col, "")
            norm = normalize_code(raw_val)
            if not is_valid_code(norm):
                continue  # skip blanks, internal codes, non-CPT/HCPCS values

            description = next(
                (row[c] for c in desc_cols if pd.notna(row.get(c))), ""
            )
            charge = next(
                (row[c] for c in charge_cols if pd.notna(row.get(c))), ""
            )

            matched_rows.append({
                "year":          year,
                "hospital":      hospital,
                "file_name":     filename,
                "sheet_name":    sheet_name,
                "source_column": code_col,
                "procedure_code": norm,
                "code_type":     detect_code_type(norm),
                "description":   str(description).strip(),
                "charge":        str(charge).strip(),
            })

    log_entry = {
        "file_name":         filename,
        "sheet_name":        sheet_name,
        "code_cols_found":   ", ".join(code_cols) if code_cols else "NONE",
        "desc_cols_found":   ", ".join(desc_cols) if desc_cols else "NONE",
        "charge_cols_found": ", ".join(charge_cols) if charge_cols else "NONE",
        "num_matches":       len(matched_rows),
    }
    return matched_rows, log_entry


# ─────────────────────────────────────────────
# FILE PROCESSING
# ─────────────────────────────────────────────

def process_file(
    filepath: str,
    year: str,
    hospital: str,
) -> tuple[list[dict], list[dict], list[dict]]:
    """
    Process one file (Excel or CSV).

    Returns:
        all_matches   – matched rows from this file
        log_entries   – one log entry per sheet/file
        error_entries – populated only on failure
    """
    filename = os.path.basename(filepath)
    all_matches, log_entries, error_entries = [], [], []

    try:
        ext = os.path.splitext(filepath)[1].lower()

        if ext in (".xlsx", ".xls", ".xlsm"):
            engine = get_engine(filepath)
            xl = pd.ExcelFile(filepath, engine=engine)
            sheets = xl.sheet_names
            for sheet in sheets:
                try:
                    df = read_sheet(filepath, sheet, engine)
                    matches, log = extract_matches(
                        df, year, hospital, filename, sheet
                    )
                    all_matches.extend(matches)
                    log_entries.append(log)
                    print(
                        f"  [sheet] {sheet:40s} → {log['num_matches']} match(es)"
                    )
                except Exception as sheet_err:
                    print(f"  [WARN] Could not read sheet '{sheet}': {sheet_err}")
                    log_entries.append({
                        "file_name": filename,
                        "sheet_name": sheet,
                        "code_cols_found": "ERROR",
                        "desc_cols_found": "",
                        "charge_cols_found": "",
                        "num_matches": 0,
                    })

        elif ext == ".csv":
            df = read_csv_file(filepath)
            matches, log = extract_matches(
                df, year, hospital, filename, "N/A"
            )
            all_matches.extend(matches)
            log_entries.append(log)
            print(f"  [csv ] {filename:40s} → {log['num_matches']} match(es)")

        else:
            print(f"  [SKIP] Unsupported file type: {filename}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {filename}: {e}")
        error_entries.append({
            "file_path": filepath,
            "year":      year,
            "hospital":  hospital,
            "error":     str(e),
        })

    return all_matches, log_entries, error_entries


# ─────────────────────────────────────────────
# DIRECTORY TRAVERSAL
# ─────────────────────────────────────────────

def traverse_and_extract(base_dir: str):
    """
    Walk base_dir/year/hospital/files and collect all results.
    Folder depth is: base_dir → year → hospital → (any depth of files).
    """
    all_matches, all_logs, all_errors = [], [], []

    if not os.path.isdir(base_dir):
        print(f"[ERROR] Base directory not found: {base_dir}")
        return all_matches, all_logs, all_errors

    # ── DEMO MODE SETTINGS ───────────────────────────────────────────────
    # Hospitals to sample per year for demo (set to None for all hospitals)
    DEMO_HOSPITALS_PER_YEAR = 3  # None = ALL hospitals, set a number e.g. 3 to limit
    # Only process 2018 and after
    # None = ALL years | customize as needed:
    #   TEST_YEARS = {"2018"}               → only 2018
    #   TEST_YEARS = {"2018", "2019"}        → custom years
    #   TEST_YEARS = None                    → all years
    TEST_YEARS = {"2024"}  # only 2024 for testing
    # ─────────────────────────────────────────────────────────────────────

    for year_entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not year_entry.is_dir():
            continue
        # Extract 4-digit year from folder name regardless of naming convention
        # Handles: '2014-hospital-chargemasters', 'chargemaster-cdm-2020r', etc.
        year_match = re.search(r'(20\d{2})', year_entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Skip years outside test range
        if TEST_YEARS and year not in TEST_YEARS:
            print(f"[SKIP] {year_entry.name}")
            continue

        # Drill down through subfolders until we find hospital folders
        # Handles: year/hospitals, year/year/hospitals, year/folder/hospitals
        def find_hospital_dir(path, depth=0):
            if depth > 3:
                return path
            subdirs = [s for s in os.scandir(path) if s.is_dir()]
            if not subdirs:
                return path
            # If subdirs contain files -> these are hospital folders
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f))
                    for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                return path
            # Otherwise go deeper
            return find_hospital_dir(subdirs[0].path, depth + 1)

        hospital_search_dir = find_hospital_dir(year_entry.path)
        print(f"[{year}] Hospital dir: {os.path.relpath(hospital_search_dir, year_entry.path)}")


        all_hospitals = sorted(
            [e for e in os.scandir(hospital_search_dir) if e.is_dir()],
            key=lambda e: e.name
        )
        # In demo mode, only take first N hospitals per year
        if DEMO_HOSPITALS_PER_YEAR:
            all_hospitals = all_hospitals[:DEMO_HOSPITALS_PER_YEAR]
            print(f"  [DEMO] Sampling {len(all_hospitals)} hospitals for {year}")

        for hospital_entry in all_hospitals:
            hospital = hospital_entry.name
            print(f"\n[{year}] {hospital}")

            # Only look at files directly inside the hospital folder (no deeper)
            # os.walk was recursing into sibling folders causing hospital mismatch
            for fname in os.listdir(hospital_entry.path):
                ext = os.path.splitext(fname)[1].lower()
                if ext not in (".xlsx", ".xls", ".xlsm", ".csv"):
                    continue
                fpath = os.path.join(hospital_entry.path, fname)
                if not os.path.isfile(fpath):
                    continue
                print(f"  Processing: {fname}")
                matches, logs, errors = process_file(
                    fpath, year, hospital
                )
                all_matches.extend(matches)
                all_logs.extend(logs)
                all_errors.extend(errors)

    return all_matches, all_logs, all_errors


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Traverse directory and extract all CPT/HCPCS codes
    print(f"\n[INFO] Scanning: {BASE_DIR}\n{'─'*60}")
    all_matches, all_logs, all_errors = traverse_and_extract(BASE_DIR)

    # 3. Save matched_rows.csv
    matched_path = os.path.join(OUTPUT_DIR, "matched_rows.csv")
    matched_cols = [
        "year", "hospital", "file_name", "sheet_name",
        "source_column", "procedure_code", "code_type",
        "description", "charge",
    ]
    if all_matches:
        pd.DataFrame(all_matches, columns=matched_cols).to_csv(
            matched_path, index=False
        )
    else:
        pd.DataFrame(columns=matched_cols).to_csv(matched_path, index=False)
    print(f"\n[OUT] matched_rows.csv          → {len(all_matches)} row(s)")

    # 4. Save extraction_log.csv
    log_path = os.path.join(OUTPUT_DIR, "extraction_log.csv")
    pd.DataFrame(all_logs).to_csv(log_path, index=False)
    print(f"[OUT] extraction_log.csv        → {len(all_logs)} sheet(s) processed")

    # 5. Save missing_files_or_errors.csv
    err_path = os.path.join(OUTPUT_DIR, "missing_files_or_errors.csv")
    pd.DataFrame(all_errors).to_csv(err_path, index=False)
    print(f"[OUT] missing_files_or_errors.csv → {len(all_errors)} error(s)")

    print("\n[DONE] Extraction complete.")


if __name__ == "__main__":
    main()


[INFO] Scanning: C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped
────────────────────────────────────────────────────────────
[SKIP] 2013-hospital-chargemasters
[SKIP] 2014-hospital-chargemasters
[SKIP] 2015-hospital-chargemasters
[SKIP] 2016-hospital-chargemasters
[SKIP] 2017-hospital-chargemasters
[SKIP] 2018-hospital-chargemasters
[SKIP] 2019-hospital-chargemasters
[SKIP] 2023-hospital-chargemaster
[SKIP] chargemaster-cdm-2020r
[SKIP] chargemaster-cdm-2021
[SKIP] chargemastercdm-2022
[2024] Hospital dir: ChargemasterCDM-2024
  [DEMO] Sampling 3 hospitals for 2024

[2024] ADVENTIST HEALTH AND RIDEOUT
  Processing: 106580996_CDM_All_AHRO_2024.xlsx
  [sheet] Cover Page                               → 0 match(es)
  [sheet] Price Change Impact                      → 0 match(es)
  [sheet] Common OP Procedures                     → 4 match(es)
  [sheet] Hospital CDM                             → 31 match(es)

[2024] ADVENTIST HEALTH BAKERSFIELD
  Processing: 106150

In [4]:
"""
validate_extraction.py
-----------------------
Validates the chargemaster extraction results by comparing:
1. How many year folders exist in the data
2. How many hospital folders exist per year
3. How many hospitals we actually got data for
4. What CPT/HCPCS codes were found and how many times
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# PATHS — update if needed
# ─────────────────────────────────────────────
BASE_DIR   = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"


# ─────────────────────────────────────────────
# STEP 1 — What exists in the folder structure?
# ─────────────────────────────────────────────
def scan_folder_structure(base_dir):
    print("\n" + "="*60)
    print("STEP 1: FOLDER STRUCTURE SUMMARY")
    print("="*60)

    year_summary = {}

    for entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not entry.is_dir():
            continue
        year_match = re.search(r'(20\d{2})', entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Drill down to find hospital folders
        hospital_dir = entry.path
        for _ in range(3):  # max 3 levels deep
            subdirs = [s for s in os.scandir(hospital_dir) if s.is_dir()]
            if not subdirs:
                break
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f)) for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                break
            hospital_dir = subdirs[0].path

        hospitals = [s.name for s in os.scandir(hospital_dir) if s.is_dir()]
        year_summary[year] = {
            "folder": entry.name,
            "hospital_count": len(hospitals),
            "hospitals": sorted(hospitals)
        }

        print(f"\n  [{year}] Folder: {entry.name}")
        print(f"         Hospitals found: {len(hospitals)}")

    print(f"\n  TOTAL YEARS IN FOLDER : {len(year_summary)}")
    print(f"  YEARS                 : {sorted(year_summary.keys())}")
    total_hospitals = sum(v['hospital_count'] for v in year_summary.values())
    print(f"  TOTAL HOSPITAL FOLDERS: {total_hospitals}")

    return year_summary


# ─────────────────────────────────────────────
# STEP 2 — What did we actually extract?
# ─────────────────────────────────────────────
def validate_extraction(output_dir, year_summary):
    print("\n" + "="*60)
    print("STEP 2: EXTRACTION RESULTS SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    log_path     = os.path.join(output_dir, "extraction_log.csv")
    error_path   = os.path.join(output_dir, "missing_files_or_errors.csv")

    if not os.path.exists(matched_path):
        print("\n  [ERROR] matched_rows.csv not found. Run the extractor first.")
        return

    df      = pd.read_csv(matched_path, dtype=str)
    log_df  = pd.read_csv(log_path, dtype=str) if os.path.exists(log_path) else pd.DataFrame()
    err_df  = pd.read_csv(error_path, dtype=str) if os.path.exists(error_path) else pd.DataFrame()

    print(f"\n  Total matched rows     : {len(df)}")
    print(f"  Total sheets processed : {len(log_df)}")
    print(f"  Total files with errors: {len(err_df)}")

    # Years extracted
    years_extracted = sorted(df["year"].dropna().unique())
    print(f"\n  Years with data        : {len(years_extracted)}")
    print(f"  Years                  : {years_extracted}")

    # Hospitals extracted per year
    print(f"\n  {'YEAR':<8} {'HOSPITALS IN FOLDER':>22} {'HOSPITALS WITH DATA':>22} {'TOTAL ROWS':>12}")
    print(f"  {'-'*68}")
    for year in sorted(df["year"].dropna().unique()):
        year_df     = df[df["year"] == year]
        got_data    = year_df["hospital"].nunique()
        in_folder   = year_summary.get(year, {}).get("hospital_count", "N/A")
        rows        = len(year_df)
        print(f"  {year:<8} {str(in_folder):>22} {got_data:>22} {rows:>12}")

    # Missing hospitals (in folder but not in output)
    print(f"\n" + "="*60)
    print("STEP 3: HOSPITALS IN FOLDER BUT NO DATA EXTRACTED")
    print("="*60)
    for year, info in sorted(year_summary.items()):
        if year not in years_extracted:
            print(f"\n  [{year}] — entire year was skipped (not in TEST_YEARS)")
            continue
        year_df         = df[df["year"] == year]
        extracted_hosp  = set(year_df["hospital"].dropna().str.strip().str.upper())
        folder_hosp     = set(h.upper() for h in info["hospitals"])
        missing         = folder_hosp - extracted_hosp
        if missing:
            print(f"\n  [{year}] {len(missing)} hospitals had no matches:")
            for h in sorted(missing)[:10]:  # show max 10
                print(f"    - {h}")
            if len(missing) > 10:
                print(f"    ... and {len(missing)-10} more")
        else:
            print(f"\n  [{year}] All hospitals had at least one match ✓")


# ─────────────────────────────────────────────
# STEP 3 — CPT / HCPCS code summary
# ─────────────────────────────────────────────
def validate_codes(output_dir):
    print(f"\n" + "="*60)
    print("STEP 4: CPT / HCPCS CODE SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    df = pd.read_csv(matched_path, dtype=str)

    # Code type split
    print(f"\n  Code type breakdown:")
    print(df["code_type"].value_counts().to_string())

    # All unique codes found
    code_counts = df.groupby(["procedure_code", "code_type"]).size().reset_index(name="occurrences")
    code_counts = code_counts.sort_values("occurrences", ascending=False)

    print(f"\n  {'CODE':<12} {'TYPE':<8} {'OCCURRENCES':>12}")
    print(f"  {'-'*35}")
    for _, row in code_counts.iterrows():
        print(f"  {row['procedure_code']:<12} {row['code_type']:<8} {row['occurrences']:>12}")

    print(f"\n  TOTAL UNIQUE CODES FOUND: {len(code_counts)}")

    # Codes in target list but NOT found anywhere
    TARGET_CODES = {
        "99281", "99282", "99283", "99284", "99285",
        "99291", "99292", "99288",
        "A0427", "A0429", "A0433",
        "57452", "57454", "57455", "57456",
        "57461", "57500", "57505",
        "58100", "58110",
        "58150", "58180", "58260", "58262", "58263",
        "58270", "58290", "58291", "58292", "58294",
        "58300", "58301", "58340", "58353", "58356",
        "58558", "58559", "58560",
        "58600", "58605", "58611",
        "58661", "58662", "58700", "58720",
        "76856", "76857", "76830", "77067",
        "59400", "59409", "59410",
        "59510", "59514", "59515",
        "59610", "59612", "59614",
    }
    found_codes  = set(df["procedure_code"].dropna().unique())
    missing_codes = TARGET_CODES - found_codes
    print(f"\n  Target codes NOT found in any file ({len(missing_codes)}):")
    for c in sorted(missing_codes):
        print(f"    - {c}")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("\n" + "="*60)
    print("   CHARGEMASTER EXTRACTION VALIDATOR")
    print("="*60)

    year_summary = scan_folder_structure(BASE_DIR)
    validate_extraction(OUTPUT_DIR, year_summary)
    validate_codes(OUTPUT_DIR)

    print("\n" + "="*60)
    print("   VALIDATION COMPLETE")
    print("="*60)

if __name__ == "__main__":
    main()


   CHARGEMASTER EXTRACTION VALIDATOR

STEP 1: FOLDER STRUCTURE SUMMARY

  [2013] Folder: 2013-hospital-chargemasters
         Hospitals found: 408

  [2014] Folder: 2014-hospital-chargemasters
         Hospitals found: 408

  [2015] Folder: 2015-hospital-chargemasters
         Hospitals found: 404

  [2016] Folder: 2016-hospital-chargemasters
         Hospitals found: 399

  [2017] Folder: 2017-hospital-chargemasters
         Hospitals found: 310

  [2018] Folder: 2018-hospital-chargemasters
         Hospitals found: 1

  [2019] Folder: 2019-hospital-chargemasters
         Hospitals found: 363

  [2023] Folder: 2023-hospital-chargemaster
         Hospitals found: 448

  [2020] Folder: chargemaster-cdm-2020r
         Hospitals found: 334

  [2021] Folder: chargemaster-cdm-2021
         Hospitals found: 318

  [2022] Folder: chargemastercdm-2022
         Hospitals found: 450

  [2024] Folder: chargemastercdm-2024
         Hospitals found: 457

  TOTAL YEARS IN FOLDER : 12
  YEARS       

EmptyDataError: No columns to parse from file

In [5]:
"""
validate_extraction.py
-----------------------
Validates the chargemaster extraction results by comparing:
1. How many year folders exist in the data
2. How many hospital folders exist per year
3. How many hospitals we actually got data for
4. What CPT/HCPCS codes were found and how many times
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# PATHS — update if needed
# ─────────────────────────────────────────────
BASE_DIR   = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"


# ─────────────────────────────────────────────
# STEP 1 — What exists in the folder structure?
# ─────────────────────────────────────────────
def scan_folder_structure(base_dir):
    print("\n" + "="*60)
    print("STEP 1: FOLDER STRUCTURE SUMMARY")
    print("="*60)

    year_summary = {}

    for entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not entry.is_dir():
            continue
        year_match = re.search(r'(20\d{2})', entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Drill down to find hospital folders
        hospital_dir = entry.path
        for _ in range(3):  # max 3 levels deep
            subdirs = [s for s in os.scandir(hospital_dir) if s.is_dir()]
            if not subdirs:
                break
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f)) for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                break
            hospital_dir = subdirs[0].path

        hospitals = [s.name for s in os.scandir(hospital_dir) if s.is_dir()]
        year_summary[year] = {
            "folder": entry.name,
            "hospital_count": len(hospitals),
            "hospitals": sorted(hospitals)
        }

        print(f"\n  [{year}] Folder: {entry.name}")
        print(f"         Hospitals found: {len(hospitals)}")

    print(f"\n  TOTAL YEARS IN FOLDER : {len(year_summary)}")
    print(f"  YEARS                 : {sorted(year_summary.keys())}")
    total_hospitals = sum(v['hospital_count'] for v in year_summary.values())
    print(f"  TOTAL HOSPITAL FOLDERS: {total_hospitals}")

    return year_summary


# ─────────────────────────────────────────────
# STEP 2 — What did we actually extract?
# ─────────────────────────────────────────────
def validate_extraction(output_dir, year_summary):
    print("\n" + "="*60)
    print("STEP 2: EXTRACTION RESULTS SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    log_path     = os.path.join(output_dir, "extraction_log.csv")
    error_path   = os.path.join(output_dir, "missing_files_or_errors.csv")

    if not os.path.exists(matched_path):
        print("\n  [ERROR] matched_rows.csv not found. Run the extractor first.")
        return

    df      = pd.read_csv(matched_path, dtype=str)
    def safe_read(path):
        try:
            return pd.read_csv(path, dtype=str)
        except Exception:
            return pd.DataFrame()

    log_df  = safe_read(log_path)
    err_df  = safe_read(error_path)

    print(f"\n  Total matched rows     : {len(df)}")
    print(f"  Total sheets processed : {len(log_df)}")
    print(f"  Total files with errors: {len(err_df)}")

    # Years extracted
    years_extracted = sorted(df["year"].dropna().unique())
    print(f"\n  Years with data        : {len(years_extracted)}")
    print(f"  Years                  : {years_extracted}")

    # Hospitals extracted per year
    print(f"\n  {'YEAR':<8} {'HOSPITALS IN FOLDER':>22} {'HOSPITALS WITH DATA':>22} {'TOTAL ROWS':>12}")
    print(f"  {'-'*68}")
    for year in sorted(df["year"].dropna().unique()):
        year_df     = df[df["year"] == year]
        got_data    = year_df["hospital"].nunique()
        in_folder   = year_summary.get(year, {}).get("hospital_count", "N/A")
        rows        = len(year_df)
        print(f"  {year:<8} {str(in_folder):>22} {got_data:>22} {rows:>12}")

    # Missing hospitals (in folder but not in output)
    print(f"\n" + "="*60)
    print("STEP 3: HOSPITALS IN FOLDER BUT NO DATA EXTRACTED")
    print("="*60)
    for year, info in sorted(year_summary.items()):
        if year not in years_extracted:
            print(f"\n  [{year}] — entire year was skipped (not in TEST_YEARS)")
            continue
        year_df         = df[df["year"] == year]
        extracted_hosp  = set(year_df["hospital"].dropna().str.strip().str.upper())
        folder_hosp     = set(h.upper() for h in info["hospitals"])
        missing         = folder_hosp - extracted_hosp
        if missing:
            print(f"\n  [{year}] {len(missing)} hospitals had no matches:")
            for h in sorted(missing)[:10]:  # show max 10
                print(f"    - {h}")
            if len(missing) > 10:
                print(f"    ... and {len(missing)-10} more")
        else:
            print(f"\n  [{year}] All hospitals had at least one match ✓")


# ─────────────────────────────────────────────
# STEP 3 — CPT / HCPCS code summary
# ─────────────────────────────────────────────
def validate_codes(output_dir):
    print(f"\n" + "="*60)
    print("STEP 4: CPT / HCPCS CODE SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    df = pd.read_csv(matched_path, dtype=str)

    # Code type split
    print(f"\n  Code type breakdown:")
    print(df["code_type"].value_counts().to_string())

    # All unique codes found
    code_counts = df.groupby(["procedure_code", "code_type"]).size().reset_index(name="occurrences")
    code_counts = code_counts.sort_values("occurrences", ascending=False)

    print(f"\n  {'CODE':<12} {'TYPE':<8} {'OCCURRENCES':>12}")
    print(f"  {'-'*35}")
    for _, row in code_counts.iterrows():
        print(f"  {row['procedure_code']:<12} {row['code_type']:<8} {row['occurrences']:>12}")

    print(f"\n  TOTAL UNIQUE CODES FOUND: {len(code_counts)}")

    # Codes in target list but NOT found anywhere
    TARGET_CODES = {
        "99281", "99282", "99283", "99284", "99285",
        "99291", "99292", "99288",
        "A0427", "A0429", "A0433",
        "57452", "57454", "57455", "57456",
        "57461", "57500", "57505",
        "58100", "58110",
        "58150", "58180", "58260", "58262", "58263",
        "58270", "58290", "58291", "58292", "58294",
        "58300", "58301", "58340", "58353", "58356",
        "58558", "58559", "58560",
        "58600", "58605", "58611",
        "58661", "58662", "58700", "58720",
        "76856", "76857", "76830", "77067",
        "59400", "59409", "59410",
        "59510", "59514", "59515",
        "59610", "59612", "59614",
    }
    found_codes  = set(df["procedure_code"].dropna().unique())
    missing_codes = TARGET_CODES - found_codes
    print(f"\n  Target codes NOT found in any file ({len(missing_codes)}):")
    for c in sorted(missing_codes):
        print(f"    - {c}")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("\n" + "="*60)
    print("   CHARGEMASTER EXTRACTION VALIDATOR")
    print("="*60)

    year_summary = scan_folder_structure(BASE_DIR)
    validate_extraction(OUTPUT_DIR, year_summary)
    validate_codes(OUTPUT_DIR)

    print("\n" + "="*60)
    print("   VALIDATION COMPLETE")
    print("="*60)

if __name__ == "__main__":
    main()


   CHARGEMASTER EXTRACTION VALIDATOR

STEP 1: FOLDER STRUCTURE SUMMARY

  [2013] Folder: 2013-hospital-chargemasters
         Hospitals found: 408

  [2014] Folder: 2014-hospital-chargemasters
         Hospitals found: 408

  [2015] Folder: 2015-hospital-chargemasters
         Hospitals found: 404

  [2016] Folder: 2016-hospital-chargemasters
         Hospitals found: 399

  [2017] Folder: 2017-hospital-chargemasters
         Hospitals found: 310

  [2018] Folder: 2018-hospital-chargemasters
         Hospitals found: 1

  [2019] Folder: 2019-hospital-chargemasters
         Hospitals found: 363

  [2023] Folder: 2023-hospital-chargemaster
         Hospitals found: 448

  [2020] Folder: chargemaster-cdm-2020r
         Hospitals found: 334

  [2021] Folder: chargemaster-cdm-2021
         Hospitals found: 318

  [2022] Folder: chargemastercdm-2022
         Hospitals found: 450

  [2024] Folder: chargemastercdm-2024
         Hospitals found: 457

  TOTAL YEARS IN FOLDER : 12
  YEARS       

In [6]:
"""
chargemaster_extractor.py
--------------------------
Recursively traverses a folder structure organized by year → hospital → files,
extracts ALL rows containing valid CPT or HCPCS codes, and writes three output CSVs.
No target list required — every valid code found is captured.

Expected layout:
    base_dir/
        2013/
            HospitalA/
                chargemaster.xlsx
            HospitalB/
                cdm.csv
        2014/
            ...

Outputs (written next to this script):
    matched_rows.csv        – all matched rows with metadata
    extraction_log.csv      – per-sheet processing summary
    missing_files_or_errors.csv – files that could not be read
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# Root directory containing year folders
BASE_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"

# Where to write the three output files
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"

# ─────────────────────────────────────────────
# TARGET CPT / HCPCS CODES
# ─────────────────────────────────────────────
# Emergency Medicine CPT codes (99281-99285 = ER visits by severity)
# Gynecology CPT codes (common OB/GYN procedures)
# Add or remove codes here as needed for your research.
TARGET_CODES = {
    # ── Emergency Medicine ──────────────────────────────────────────
    "99281", "99282", "99283", "99284", "99285",  # ER visit levels 1-5
    "99291", "99292",                              # Critical care

    # ── Gynecology ──────────────────────────────────────────────────
    "59400", "59409", "59410",                     # Vaginal delivery
    "59510", "59514", "59515",                     # Cesarean delivery
}

# ─────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────

# Column-name patterns that likely contain procedure codes
CODE_COL_PATTERNS = re.compile(
    r"(cpt|hcpcs|procedure\s*code|proc\s*code|cpt[\s/\-]*hcpcs|service\s*code|"
    r"\bcode\b|billing\s*code|revenue\s*code)",
    re.IGNORECASE,
)

# Column-name patterns for description and charge
DESC_COL_PATTERNS = re.compile(
    r"(description|desc|procedure\s*name|service\s*name|item\s*name|narrative)",
    re.IGNORECASE,
)
CHARGE_COL_PATTERNS = re.compile(
    # Match price/amount columns but EXCLUDE "Charge Code" (internal hospital ID)
    # Also catches "June 2024 Prices", "Average Charge", "Gross Charge" etc.
    r"(?i)^(?!charge\s*code)(?=.*(price|amount|rate|average\s*charge|avg\s*charge|"
    r"gross\s*charge|standard\s*charge|billed\s*charge|cdm\s*price|list\s*price|"
    r"\d{4}\s*price))",
    re.IGNORECASE,
)

# CPT codes: 5 digits (optionally followed by a 2-char modifier)
CPT_PATTERN = re.compile(r"^\d{5}([A-Z0-9]{2})?$")

# HCPCS Level II codes: letter + 4 digits
HCPCS_PATTERN = re.compile(r"^[A-Z]\d{4}$")


# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────


def normalize_code(value) -> str:
    """Strip whitespace, uppercase, remove non-alphanumeric characters."""
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())


def detect_code_type(code: str) -> str:
    """Return 'CPT', 'HCPCS', or 'UNKNOWN' based on code format."""
    if CPT_PATTERN.match(code):
        return "CPT"
    if HCPCS_PATTERN.match(code):
        return "HCPCS"
    return "UNKNOWN"


def find_columns(df: pd.DataFrame, pattern: re.Pattern) -> list:
    """Return column names matching a regex pattern."""
    return [c for c in df.columns if pattern.search(str(c))]


# Keywords that strongly indicate a real header row
HEADER_KEYWORDS = re.compile(
    r"(cpt|hcpcs|code|charge|price|amount|description|desc|procedure|service|rate|cost|revenue)",
    re.IGNORECASE,
)

def find_header_row(df_raw: pd.DataFrame, max_scan: int = 15) -> int:
    """
    Find the real header row by scoring each row on two criteria:
    1. KEYWORD score  — how many cells contain header-like words (cpt, charge, description etc.)
    2. BREADTH score  — how many non-empty string cells (width of the header)
    Keyword hits are weighted heavily so title rows like "Common OP Procedures"
    don't outscore a real multi-column header.
    """
    best_row, best_score = 0, -1
    for i in range(min(max_scan, len(df_raw))):
        row = df_raw.iloc[i]
        keyword_hits = sum(
            1 for v in row
            if isinstance(v, str) and HEADER_KEYWORDS.search(v)
        )
        breadth = sum(
            1 for v in row
            if isinstance(v, str) and len(v.strip()) > 1
        )
        # Keyword hits worth 5x more than plain text breadth
        score = keyword_hits * 5 + breadth
        if score > best_score:
            best_score, best_row = score, i
    return best_row


def get_engine(filepath: str) -> str:
    """Return the correct pandas engine based on file extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".xls":
        return "xlrd"
    return "openpyxl"


def read_sheet(filepath: str, sheet_name, engine: str) -> pd.DataFrame:
    """
    Read a single Excel sheet in one pass, auto-detecting the header row.
    Uses xlrd for .xls and openpyxl for .xlsx/.xlsm.
    """
    # Single raw read to find header row and return data
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None,
                        dtype=str, engine=engine)
    header_row = find_header_row(raw)
    # Slice: rows above header become the header, rows below are data
    df = raw.iloc[header_row + 1:].copy()
    df.columns = [str(v).strip() for v in raw.iloc[header_row]]
    df.reset_index(drop=True, inplace=True)
    return df


def read_csv_file(filepath: str) -> pd.DataFrame:
    """Read a CSV, trying common encodings."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(filepath, dtype=str, encoding=enc)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode CSV: {filepath}")


# ─────────────────────────────────────────────
# CORE EXTRACTION LOGIC
# ─────────────────────────────────────────────

def is_valid_code(code: str) -> bool:
    """Return True if code is a valid CPT/HCPCS format AND in our target list."""
    is_valid_format = bool(CPT_PATTERN.match(code) or HCPCS_PATTERN.match(code))
    return is_valid_format and code in TARGET_CODES


def extract_matches(
    df: pd.DataFrame,
    year: str,
    hospital: str,
    filename: str,
    sheet_name: str,
) -> tuple[list[dict], dict]:
    """
    Scan `df` and extract ALL rows that contain a valid CPT or HCPCS code.

    Returns:
        matched_rows  – list of dicts (one per matched row)
        log_entry     – dict summarising what was detected in this sheet
    """
    # Identify candidate code columns by name
    code_cols = find_columns(df, CODE_COL_PATTERNS)
    desc_cols = find_columns(df, DESC_COL_PATTERNS)
    charge_cols = find_columns(df, CHARGE_COL_PATTERNS)

    # If no named code column found, scan ALL columns for target code values
    if not code_cols:
        for col in df.columns:
            sample = df[col].dropna().head(100)
            hits = sample.apply(lambda v: is_valid_code(normalize_code(v))).sum()
            if hits >= 1:  # even 1 target code hit is enough
                code_cols.append(col)

    matched_rows = []

    for code_col in code_cols:
        for _, row in df.iterrows():
            raw_val = row.get(code_col, "")
            norm = normalize_code(raw_val)
            if not is_valid_code(norm):
                continue  # skip blanks, internal codes, non-CPT/HCPCS values

            description = next(
                (row[c] for c in desc_cols if pd.notna(row.get(c))), ""
            )
            charge = next(
                (row[c] for c in charge_cols if pd.notna(row.get(c))), ""
            )

            matched_rows.append({
                "year":          year,
                "hospital":      hospital,
                "file_name":     filename,
                "sheet_name":    sheet_name,
                "source_column": code_col,
                "procedure_code": norm,
                "code_type":     detect_code_type(norm),
                "description":   str(description).strip(),
                "charge":        str(charge).strip(),
            })

    log_entry = {
        "file_name":         filename,
        "sheet_name":        sheet_name,
        "code_cols_found":   ", ".join(code_cols) if code_cols else "NONE",
        "desc_cols_found":   ", ".join(desc_cols) if desc_cols else "NONE",
        "charge_cols_found": ", ".join(charge_cols) if charge_cols else "NONE",
        "num_matches":       len(matched_rows),
    }
    return matched_rows, log_entry


# ─────────────────────────────────────────────
# FILE PROCESSING
# ─────────────────────────────────────────────

def process_file(
    filepath: str,
    year: str,
    hospital: str,
) -> tuple[list[dict], list[dict], list[dict]]:
    """
    Process one file (Excel or CSV).

    Returns:
        all_matches   – matched rows from this file
        log_entries   – one log entry per sheet/file
        error_entries – populated only on failure
    """
    filename = os.path.basename(filepath)
    all_matches, log_entries, error_entries = [], [], []

    try:
        ext = os.path.splitext(filepath)[1].lower()

        if ext in (".xlsx", ".xls", ".xlsm"):
            engine = get_engine(filepath)
            xl = pd.ExcelFile(filepath, engine=engine)
            sheets = xl.sheet_names
            for sheet in sheets:
                try:
                    df = read_sheet(filepath, sheet, engine)
                    matches, log = extract_matches(
                        df, year, hospital, filename, sheet
                    )
                    all_matches.extend(matches)
                    log_entries.append(log)
                    print(
                        f"  [sheet] {sheet:40s} → {log['num_matches']} match(es)"
                    )
                except Exception as sheet_err:
                    print(f"  [WARN] Could not read sheet '{sheet}': {sheet_err}")
                    log_entries.append({
                        "file_name": filename,
                        "sheet_name": sheet,
                        "code_cols_found": "ERROR",
                        "desc_cols_found": "",
                        "charge_cols_found": "",
                        "num_matches": 0,
                    })

        elif ext == ".csv":
            df = read_csv_file(filepath)
            matches, log = extract_matches(
                df, year, hospital, filename, "N/A"
            )
            all_matches.extend(matches)
            log_entries.append(log)
            print(f"  [csv ] {filename:40s} → {log['num_matches']} match(es)")

        else:
            print(f"  [SKIP] Unsupported file type: {filename}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {filename}: {e}")
        error_entries.append({
            "file_path": filepath,
            "year":      year,
            "hospital":  hospital,
            "error":     str(e),
        })

    return all_matches, log_entries, error_entries


# ─────────────────────────────────────────────
# DIRECTORY TRAVERSAL
# ─────────────────────────────────────────────

def traverse_and_extract(base_dir: str):
    """
    Walk base_dir/year/hospital/files and collect all results.
    Folder depth is: base_dir → year → hospital → (any depth of files).
    """
    all_matches, all_logs, all_errors = [], [], []

    if not os.path.isdir(base_dir):
        print(f"[ERROR] Base directory not found: {base_dir}")
        return all_matches, all_logs, all_errors

    # ── DEMO MODE SETTINGS ───────────────────────────────────────────────
    # Hospitals to sample per year for demo (set to None for all hospitals)
    DEMO_HOSPITALS_PER_YEAR = None  # None = ALL hospitals, set a number e.g. 3 to limit
    # Only process 2018 and after
    # None = ALL years | customize as needed:
    #   TEST_YEARS = {"2018"}               → only 2018
    #   TEST_YEARS = {"2018", "2019"}        → custom years
    #   TEST_YEARS = None                    → all years
    TEST_YEARS = {"2024"}  # only 2024 for testing
    # ─────────────────────────────────────────────────────────────────────

    for year_entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not year_entry.is_dir():
            continue
        # Extract 4-digit year from folder name regardless of naming convention
        # Handles: '2014-hospital-chargemasters', 'chargemaster-cdm-2020r', etc.
        year_match = re.search(r'(20\d{2})', year_entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Skip years outside test range
        if TEST_YEARS and year not in TEST_YEARS:
            print(f"[SKIP] {year_entry.name}")
            continue

        # Drill down through subfolders until we find hospital folders
        # Handles: year/hospitals, year/year/hospitals, year/folder/hospitals
        def find_hospital_dir(path, depth=0):
            if depth > 3:
                return path
            subdirs = [s for s in os.scandir(path) if s.is_dir()]
            if not subdirs:
                return path
            # If subdirs contain files -> these are hospital folders
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f))
                    for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                return path
            # Otherwise go deeper
            return find_hospital_dir(subdirs[0].path, depth + 1)

        hospital_search_dir = find_hospital_dir(year_entry.path)
        print(f"[{year}] Hospital dir: {os.path.relpath(hospital_search_dir, year_entry.path)}")


        all_hospitals = sorted(
            [e for e in os.scandir(hospital_search_dir) if e.is_dir()],
            key=lambda e: e.name
        )
        # In demo mode, only take first N hospitals per year
        if DEMO_HOSPITALS_PER_YEAR:
            all_hospitals = all_hospitals[:DEMO_HOSPITALS_PER_YEAR]
            print(f"  [DEMO] Sampling {len(all_hospitals)} hospitals for {year}")

        for hospital_entry in all_hospitals:
            hospital = hospital_entry.name
            print(f"\n[{year}] {hospital}")

            # Only look at files directly inside the hospital folder (no deeper)
            # os.walk was recursing into sibling folders causing hospital mismatch
            for fname in os.listdir(hospital_entry.path):
                ext = os.path.splitext(fname)[1].lower()
                if ext not in (".xlsx", ".xls", ".xlsm", ".csv"):
                    continue
                fpath = os.path.join(hospital_entry.path, fname)
                if not os.path.isfile(fpath):
                    continue
                print(f"  Processing: {fname}")
                matches, logs, errors = process_file(
                    fpath, year, hospital
                )
                all_matches.extend(matches)
                all_logs.extend(logs)
                all_errors.extend(errors)

    return all_matches, all_logs, all_errors


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Traverse directory and extract all CPT/HCPCS codes
    print(f"\n[INFO] Scanning: {BASE_DIR}\n{'─'*60}")
    all_matches, all_logs, all_errors = traverse_and_extract(BASE_DIR)

    # 3. Save matched_rows.csv
    matched_path = os.path.join(OUTPUT_DIR, "matched_rows.csv")
    matched_cols = [
        "year", "hospital", "file_name", "sheet_name",
        "source_column", "procedure_code", "code_type",
        "description", "charge",
    ]
    if all_matches:
        pd.DataFrame(all_matches, columns=matched_cols).to_csv(
            matched_path, index=False
        )
    else:
        pd.DataFrame(columns=matched_cols).to_csv(matched_path, index=False)
    print(f"\n[OUT] matched_rows.csv          → {len(all_matches)} row(s)")

    # 4. Save extraction_log.csv
    log_path = os.path.join(OUTPUT_DIR, "extraction_log.csv")
    pd.DataFrame(all_logs).to_csv(log_path, index=False)
    print(f"[OUT] extraction_log.csv        → {len(all_logs)} sheet(s) processed")

    # 5. Save missing_files_or_errors.csv
    err_path = os.path.join(OUTPUT_DIR, "missing_files_or_errors.csv")
    pd.DataFrame(all_errors).to_csv(err_path, index=False)
    print(f"[OUT] missing_files_or_errors.csv → {len(all_errors)} error(s)")

    print("\n[DONE] Extraction complete.")


if __name__ == "__main__":
    main()


[INFO] Scanning: C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped
────────────────────────────────────────────────────────────
[SKIP] 2013-hospital-chargemasters
[SKIP] 2014-hospital-chargemasters
[SKIP] 2015-hospital-chargemasters
[SKIP] 2016-hospital-chargemasters
[SKIP] 2017-hospital-chargemasters
[SKIP] 2018-hospital-chargemasters
[SKIP] 2019-hospital-chargemasters
[SKIP] 2023-hospital-chargemaster
[SKIP] chargemaster-cdm-2020r
[SKIP] chargemaster-cdm-2021
[SKIP] chargemastercdm-2022
[2024] Hospital dir: ChargemasterCDM-2024

[2024] ADVENTIST HEALTH AND RIDEOUT
  Processing: 106580996_CDM_All_AHRO_2024.xlsx
  [sheet] Cover Page                               → 0 match(es)
  [sheet] Price Change Impact                      → 0 match(es)
  [sheet] Common OP Procedures                     → 4 match(es)
  [sheet] Hospital CDM                             → 31 match(es)

[2024] ADVENTIST HEALTH BAKERSFIELD
  Processing: 106150788_CDM_All_AHBD_2024.xlsx
  [sheet] Co

C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


  [sheet] CDM_Summit 2024_06_01                    → 0 match(es)
  [sheet] Rx Download 2024.06.01                   → 0 match(es)
  [sheet] Est Perct Chg GR                         → 0 match(es)

[2024] ALTA BATES SUMMIT MEDICAL CENTER - SUMMIT CAMPUS
  Processing: 106013626_CDM_ALL_2024_Summit_Medical_Center.xlsx
  [sheet] AB 1045 FORM (Merritt Campus)            → 3 match(es)
  [sheet] AB 1045 FORM (Provd Campus)              → 3 match(es)


C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


  [sheet] CDM_Summit 2024_06_01                    → 0 match(es)
  [sheet] Rx Download 2024.06.01                   → 0 match(es)
  [sheet] Est Perct Chg GR                         → 0 match(es)

[2024] ALTA BATES SUMMIT MEDICAL CENTER-ALTA BATES CAMPUS
  Processing: 106010739_CDM_ALL_2024_Alta_Bates.xlsx
  [sheet] AB 1045 FORM  (Ashby Campus)             → 3 match(es)
  [sheet] AB 1045 FORM (Herrick Campus)            → 3 match(es)
  [sheet] CDM ALTA BATES 2024_06_01                → 0 match(es)
  [sheet] Rx Download 2024_06_01                   → 0 match(es)
  [sheet] Alta Bates Rate Increase                 → 0 match(es)

[2024] ALTA BATES SUMMIT MEDICAL CENTER-HERRICK CAMPUS
  Processing: 106010844_CDM_ALL_2024_Alta_Bates.xlsx
  [sheet] AB 1045 FORM  (Ashby Campus)             → 3 match(es)
  [sheet] AB 1045 FORM (Herrick Campus)            → 3 match(es)
  [sheet] CDM ALTA BATES 2024_06_01                → 0 match(es)
  [sheet] Rx Download 2024_06_01                   → 0 match(es)

...

C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: '106301297_CDM'!$A:$F.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: '106301297_Common 25'!$A:$C.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: '106301297_PCT_CHG'!$A:$K.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [sheet] 106301297_Common 25                      → 4 match(es)


C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: '106301297_CDM'!$A:$F.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: '106301297_Common 25'!$A:$C.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: '106301297_PCT_CHG'!$A:$K.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [sheet] 106301297_PCT_CHG                        → 7 match(es)

[2024] PLUMAS DISTRICT HOSPITAL
  Processing: 106320986_CDM_ALLv2.xlsx
  [sheet] Top 50 List                              → 0 match(es)
  [sheet] AB 1045 Form                             → 4 match(es)
  [sheet] CDM                                      → 0 match(es)
  [sheet] Gross Revenue Percent Change             → 0 match(es)

[2024] POMONA VALLEY HOSPITAL MEDICAL CENTER
  Processing: 106190630_CDM_2024_0601.xlsx
  [sheet] Sheet1                                   → 10 match(es)
  Processing: 106190630_PCT_CHG_2024.xls
  [sheet] Sheet1                                   → 0 match(es)
  Processing: 25CommonOPProcedures2024.xlsx
  [sheet] Top 50 List                              → 0 match(es)
  [sheet] AB 1045 Form                             → 4 match(es)

[2024] PORTERVILLE DEVELOPMENTAL CENTER
  Processing: 106541123_CDM_All.xlsx
  [sheet] Chargemaster                             → 0 match(es)
  [sheet] 25 Common OP   


KeyboardInterrupt



In [1]:
"""
chargemaster_extractor.py
--------------------------
Recursively traverses a folder structure organized by year → hospital → files,
extracts ALL rows containing valid CPT or HCPCS codes, and writes three output CSVs.
No target list required — every valid code found is captured.

Expected layout:
    base_dir/
        2013/
            HospitalA/
                chargemaster.xlsx
            HospitalB/
                cdm.csv
        2014/
            ...

Outputs (written next to this script):
    matched_rows.csv        – all matched rows with metadata
    extraction_log.csv      – per-sheet processing summary
    missing_files_or_errors.csv – files that could not be read
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# Root directory containing year folders
BASE_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"

# Where to write the three output files
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"

# ─────────────────────────────────────────────
# TARGET CPT / HCPCS CODES
# ─────────────────────────────────────────────
# Emergency Medicine CPT codes (99281-99285 = ER visits by severity)
# Gynecology CPT codes (common OB/GYN procedures)
# Add or remove codes here as needed for your research.
TARGET_CODES = {
    # ── Emergency Medicine ──────────────────────────────────────────
    "99281", "99282", "99283", "99284", "99285",  # ER visit levels 1-5
    "99291", "99292",                              # Critical care

    # ── Gynecology ──────────────────────────────────────────────────
    "59400", "59409", "59410",                     # Vaginal delivery
    "59510", "59514", "59515",                     # Cesarean delivery
}

# ─────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────

# Column-name patterns that likely contain procedure codes
CODE_COL_PATTERNS = re.compile(
    r"(cpt|hcpcs|procedure\s*code|proc\s*code|cpt[\s/\-]*hcpcs|service\s*code|"
    r"\bcode\b|billing\s*code|revenue\s*code)",
    re.IGNORECASE,
)

# Column-name patterns for description and charge
DESC_COL_PATTERNS = re.compile(
    r"(description|desc|procedure\s*name|service\s*name|item\s*name|narrative)",
    re.IGNORECASE,
)
CHARGE_COL_PATTERNS = re.compile(
    # Match price/amount columns but EXCLUDE "Charge Code" (internal hospital ID)
    # Also catches "June 2024 Prices", "Average Charge", "Gross Charge" etc.
    r"(?i)^(?!charge\s*code)(?=.*(price|amount|rate|average\s*charge|avg\s*charge|"
    r"gross\s*charge|standard\s*charge|billed\s*charge|cdm\s*price|list\s*price|"
    r"\d{4}\s*price))",
    re.IGNORECASE,
)

# CPT codes: 5 digits (optionally followed by a 2-char modifier)
CPT_PATTERN = re.compile(r"^\d{5}([A-Z0-9]{2})?$")

# HCPCS Level II codes: letter + 4 digits
HCPCS_PATTERN = re.compile(r"^[A-Z]\d{4}$")


# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────


def normalize_code(value) -> str:
    """Strip whitespace, uppercase, remove non-alphanumeric characters."""
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())


def detect_code_type(code: str) -> str:
    """Return 'CPT', 'HCPCS', or 'UNKNOWN' based on code format."""
    if CPT_PATTERN.match(code):
        return "CPT"
    if HCPCS_PATTERN.match(code):
        return "HCPCS"
    return "UNKNOWN"


def find_columns(df: pd.DataFrame, pattern: re.Pattern) -> list:
    """Return column names matching a regex pattern."""
    return [c for c in df.columns if pattern.search(str(c))]


# Keywords that strongly indicate a real header row
HEADER_KEYWORDS = re.compile(
    r"(cpt|hcpcs|code|charge|price|amount|description|desc|procedure|service|rate|cost|revenue)",
    re.IGNORECASE,
)

def find_header_row(df_raw: pd.DataFrame, max_scan: int = 15) -> int:
    """
    Find the real header row by scoring each row on two criteria:
    1. KEYWORD score  — how many cells contain header-like words (cpt, charge, description etc.)
    2. BREADTH score  — how many non-empty string cells (width of the header)
    Keyword hits are weighted heavily so title rows like "Common OP Procedures"
    don't outscore a real multi-column header.
    """
    best_row, best_score = 0, -1
    for i in range(min(max_scan, len(df_raw))):
        row = df_raw.iloc[i]
        keyword_hits = sum(
            1 for v in row
            if isinstance(v, str) and HEADER_KEYWORDS.search(v)
        )
        breadth = sum(
            1 for v in row
            if isinstance(v, str) and len(v.strip()) > 1
        )
        # Keyword hits worth 5x more than plain text breadth
        score = keyword_hits * 5 + breadth
        if score > best_score:
            best_score, best_row = score, i
    return best_row


def get_engine(filepath: str) -> str:
    """Return the correct pandas engine based on file extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".xls":
        return "xlrd"
    return "openpyxl"


def read_sheet(filepath: str, sheet_name, engine: str) -> pd.DataFrame:
    """
    Read a single Excel sheet in one pass, auto-detecting the header row.
    Uses xlrd for .xls and openpyxl for .xlsx/.xlsm.
    """
    # Single raw read to find header row and return data
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None,
                        dtype=str, engine=engine)
    header_row = find_header_row(raw)
    # Slice: rows above header become the header, rows below are data
    df = raw.iloc[header_row + 1:].copy()
    df.columns = [str(v).strip() for v in raw.iloc[header_row]]
    df.reset_index(drop=True, inplace=True)
    return df


def read_csv_file(filepath: str) -> pd.DataFrame:
    """Read a CSV, trying common encodings."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(filepath, dtype=str, encoding=enc)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode CSV: {filepath}")


# ─────────────────────────────────────────────
# CORE EXTRACTION LOGIC
# ─────────────────────────────────────────────

def is_valid_code(code: str) -> bool:
    """Return True if code is a valid CPT/HCPCS format AND in our target list."""
    is_valid_format = bool(CPT_PATTERN.match(code) or HCPCS_PATTERN.match(code))
    return is_valid_format and code in TARGET_CODES


def extract_matches(
    df: pd.DataFrame,
    year: str,
    hospital: str,
    filename: str,
    sheet_name: str,
) -> tuple[list[dict], dict]:
    """
    Scan `df` and extract ALL rows that contain a valid CPT or HCPCS code.

    Returns:
        matched_rows  – list of dicts (one per matched row)
        log_entry     – dict summarising what was detected in this sheet
    """
    # Identify candidate code columns by name
    code_cols = find_columns(df, CODE_COL_PATTERNS)
    desc_cols = find_columns(df, DESC_COL_PATTERNS)
    charge_cols = find_columns(df, CHARGE_COL_PATTERNS)

    # If no named code column found, scan ALL columns for target code values
    if not code_cols:
        for col in df.columns:
            sample = df[col].dropna().head(100)
            hits = sample.apply(lambda v: is_valid_code(normalize_code(v))).sum()
            if hits >= 1:  # even 1 target code hit is enough
                code_cols.append(col)

    matched_rows = []

    for code_col in code_cols:
        for _, row in df.iterrows():
            raw_val = row.get(code_col, "")
            norm = normalize_code(raw_val)
            if not is_valid_code(norm):
                continue  # skip blanks, internal codes, non-CPT/HCPCS values

            description = next(
                (row[c] for c in desc_cols if pd.notna(row.get(c))), ""
            )
            charge = next(
                (row[c] for c in charge_cols if pd.notna(row.get(c))), ""
            )

            matched_rows.append({
                "year":          year,
                "hospital":      hospital,
                "file_name":     filename,
                "sheet_name":    sheet_name,
                "source_column": code_col,
                "procedure_code": norm,
                "code_type":     detect_code_type(norm),
                "description":   str(description).strip(),
                "charge":        str(charge).strip(),
            })

    log_entry = {
        "file_name":         filename,
        "sheet_name":        sheet_name,
        "code_cols_found":   ", ".join(code_cols) if code_cols else "NONE",
        "desc_cols_found":   ", ".join(desc_cols) if desc_cols else "NONE",
        "charge_cols_found": ", ".join(charge_cols) if charge_cols else "NONE",
        "num_matches":       len(matched_rows),
    }
    return matched_rows, log_entry


# ─────────────────────────────────────────────
# FILE PROCESSING
# ─────────────────────────────────────────────

def process_file(
    filepath: str,
    year: str,
    hospital: str,
) -> tuple[list[dict], list[dict], list[dict]]:
    """
    Process one file (Excel or CSV).

    Returns:
        all_matches   – matched rows from this file
        log_entries   – one log entry per sheet/file
        error_entries – populated only on failure
    """
    filename = os.path.basename(filepath)
    all_matches, log_entries, error_entries = [], [], []

    try:
        ext = os.path.splitext(filepath)[1].lower()

        if ext in (".xlsx", ".xls", ".xlsm"):
            engine = get_engine(filepath)
            xl = pd.ExcelFile(filepath, engine=engine)
            sheets = xl.sheet_names
            for sheet in sheets:
                try:
                    df = read_sheet(filepath, sheet, engine)
                    matches, log = extract_matches(
                        df, year, hospital, filename, sheet
                    )
                    all_matches.extend(matches)
                    log_entries.append(log)
                    print(
                        f"  [sheet] {sheet:40s} → {log['num_matches']} match(es)"
                    )
                except Exception as sheet_err:
                    print(f"  [WARN] Could not read sheet '{sheet}': {sheet_err}")
                    log_entries.append({
                        "file_name": filename,
                        "sheet_name": sheet,
                        "code_cols_found": "ERROR",
                        "desc_cols_found": "",
                        "charge_cols_found": "",
                        "num_matches": 0,
                    })

        elif ext == ".csv":
            df = read_csv_file(filepath)
            matches, log = extract_matches(
                df, year, hospital, filename, "N/A"
            )
            all_matches.extend(matches)
            log_entries.append(log)
            print(f"  [csv ] {filename:40s} → {log['num_matches']} match(es)")

        else:
            print(f"  [SKIP] Unsupported file type: {filename}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {filename}: {e}")
        error_entries.append({
            "file_path": filepath,
            "year":      year,
            "hospital":  hospital,
            "error":     str(e),
        })

    return all_matches, log_entries, error_entries


# ─────────────────────────────────────────────
# DIRECTORY TRAVERSAL
# ─────────────────────────────────────────────

def traverse_and_extract(base_dir: str):
    """
    Walk base_dir/year/hospital/files and collect all results.
    Folder depth is: base_dir → year → hospital → (any depth of files).
    """
    all_matches, all_logs, all_errors = [], [], []

    if not os.path.isdir(base_dir):
        print(f"[ERROR] Base directory not found: {base_dir}")
        return all_matches, all_logs, all_errors

    # ── DEMO MODE SETTINGS ───────────────────────────────────────────────
    # Hospitals to sample per year for demo (set to None for all hospitals)
    DEMO_HOSPITALS_PER_YEAR = None  # None = ALL hospitals, set a number e.g. 3 to limit
    # Only process 2018 and after
    # None = ALL years | customize as needed:
    #   TEST_YEARS = {"2018"}               → only 2018
    #   TEST_YEARS = {"2018", "2019"}        → custom years
    #   TEST_YEARS = None                    → all years
    TEST_YEARS = {"2024"}  # only 2024 for testing
    # ─────────────────────────────────────────────────────────────────────

    for year_entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not year_entry.is_dir():
            continue
        # Extract 4-digit year from folder name regardless of naming convention
        # Handles: '2014-hospital-chargemasters', 'chargemaster-cdm-2020r', etc.
        year_match = re.search(r'(20\d{2})', year_entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Skip years outside test range
        if TEST_YEARS and year not in TEST_YEARS:
            print(f"[SKIP] {year_entry.name}")
            continue

        # Drill down through subfolders until we find hospital folders
        # Handles: year/hospitals, year/year/hospitals, year/folder/hospitals
        def find_hospital_dir(path, depth=0):
            if depth > 3:
                return path
            subdirs = [s for s in os.scandir(path) if s.is_dir()]
            if not subdirs:
                return path
            # If subdirs contain files -> these are hospital folders
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f))
                    for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                return path
            # Otherwise go deeper
            return find_hospital_dir(subdirs[0].path, depth + 1)

        hospital_search_dir = find_hospital_dir(year_entry.path)
        print(f"[{year}] Hospital dir: {os.path.relpath(hospital_search_dir, year_entry.path)}")


        all_hospitals = sorted(
            [e for e in os.scandir(hospital_search_dir) if e.is_dir()],
            key=lambda e: e.name
        )
        # In demo mode, only take first N hospitals per year
        if DEMO_HOSPITALS_PER_YEAR:
            all_hospitals = all_hospitals[:DEMO_HOSPITALS_PER_YEAR]
            print(f"  [DEMO] Sampling {len(all_hospitals)} hospitals for {year}")

        for hospital_entry in all_hospitals:
            hospital = hospital_entry.name
            print(f"\n[{year}] {hospital}")

            # Only look at files directly inside the hospital folder (no deeper)
            # os.walk was recursing into sibling folders causing hospital mismatch
            for fname in os.listdir(hospital_entry.path):
                ext = os.path.splitext(fname)[1].lower()
                if ext not in (".xlsx", ".xls", ".xlsm", ".csv"):
                    continue
                fpath = os.path.join(hospital_entry.path, fname)
                if not os.path.isfile(fpath):
                    continue
                print(f"  Processing: {fname}")
                matches, logs, errors = process_file(
                    fpath, year, hospital
                )
                all_matches.extend(matches)
                all_logs.extend(logs)
                all_errors.extend(errors)

    return all_matches, all_logs, all_errors


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Traverse directory and extract all CPT/HCPCS codes
    print(f"\n[INFO] Scanning: {BASE_DIR}\n{'─'*60}")
    all_matches, all_logs, all_errors = traverse_and_extract(BASE_DIR)

    # 3. Save matched_rows.csv
    matched_path = os.path.join(OUTPUT_DIR, "matched_rows.csv")
    matched_cols = [
        "year", "hospital", "file_name", "sheet_name",
        "source_column", "procedure_code", "code_type",
        "description", "charge",
    ]
    if all_matches:
        pd.DataFrame(all_matches, columns=matched_cols).to_csv(
            matched_path, index=False
        )
    else:
        pd.DataFrame(columns=matched_cols).to_csv(matched_path, index=False)
    print(f"\n[OUT] matched_rows.csv          → {len(all_matches)} row(s)")

    # 4. Save extraction_log.csv
    log_path = os.path.join(OUTPUT_DIR, "extraction_log.csv")
    pd.DataFrame(all_logs).to_csv(log_path, index=False)
    print(f"[OUT] extraction_log.csv        → {len(all_logs)} sheet(s) processed")

    # 5. Save missing_files_or_errors.csv
    err_path = os.path.join(OUTPUT_DIR, "missing_files_or_errors.csv")
    pd.DataFrame(all_errors).to_csv(err_path, index=False)
    print(f"[OUT] missing_files_or_errors.csv → {len(all_errors)} error(s)")

    print("\n[DONE] Extraction complete.")


if __name__ == "__main__":
    main()


[INFO] Scanning: C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped
────────────────────────────────────────────────────────────
[SKIP] 2013-hospital-chargemasters
[SKIP] 2014-hospital-chargemasters
[SKIP] 2015-hospital-chargemasters
[SKIP] 2016-hospital-chargemasters
[SKIP] 2017-hospital-chargemasters
[SKIP] 2018-hospital-chargemasters
[SKIP] 2019-hospital-chargemasters
[SKIP] 2023-hospital-chargemaster
[SKIP] chargemaster-cdm-2020r
[SKIP] chargemaster-cdm-2021
[SKIP] chargemastercdm-2022
[2024] Hospital dir: ChargemasterCDM-2024

[2024] ADVENTIST HEALTH AND RIDEOUT
  Processing: 106580996_CDM_All_AHRO_2024.xlsx
  [sheet] Cover Page                               → 0 match(es)
  [sheet] Price Change Impact                      → 0 match(es)
  [sheet] Common OP Procedures                     → 4 match(es)
  [sheet] Hospital CDM                             → 31 match(es)

[2024] ADVENTIST HEALTH BAKERSFIELD
  Processing: 106150788_CDM_All_AHBD_2024.xlsx
  [sheet] Co

C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


  [sheet] CDM_Summit 2024_06_01                    → 0 match(es)
  [sheet] Rx Download 2024.06.01                   → 0 match(es)
  [sheet] Est Perct Chg GR                         → 0 match(es)

[2024] ALTA BATES SUMMIT MEDICAL CENTER - SUMMIT CAMPUS
  Processing: 106013626_CDM_ALL_2024_Summit_Medical_Center.xlsx
  [sheet] AB 1045 FORM (Merritt Campus)            → 3 match(es)
  [sheet] AB 1045 FORM (Provd Campus)              → 3 match(es)


C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


  [sheet] CDM_Summit 2024_06_01                    → 0 match(es)
  [sheet] Rx Download 2024.06.01                   → 0 match(es)
  [sheet] Est Perct Chg GR                         → 0 match(es)

[2024] ALTA BATES SUMMIT MEDICAL CENTER-ALTA BATES CAMPUS
  Processing: 106010739_CDM_ALL_2024_Alta_Bates.xlsx
  [sheet] AB 1045 FORM  (Ashby Campus)             → 3 match(es)
  [sheet] AB 1045 FORM (Herrick Campus)            → 3 match(es)
  [sheet] CDM ALTA BATES 2024_06_01                → 0 match(es)
  [sheet] Rx Download 2024_06_01                   → 0 match(es)
  [sheet] Alta Bates Rate Increase                 → 0 match(es)

[2024] ALTA BATES SUMMIT MEDICAL CENTER-HERRICK CAMPUS
  Processing: 106010844_CDM_ALL_2024_Alta_Bates.xlsx
  [sheet] AB 1045 FORM  (Ashby Campus)             → 3 match(es)
  [sheet] AB 1045 FORM (Herrick Campus)            → 3 match(es)
  [sheet] CDM ALTA BATES 2024_06_01                → 0 match(es)
  [sheet] Rx Download 2024_06_01                   → 0 match(es)

...

  [sheet] Price Change Impact                      → 0 match(es)


C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Hospital CDM'!$A:$F.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Common OP Procedures'!$A:$C.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Price Change Impact'!$A:$K.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [sheet] Common OP Procedures                     → 4 match(es)


C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Hospital CDM'!$A:$F.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Common OP Procedures'!$A:$C.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
C:\Users\deeksha\anaconda3\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Price Change Impact'!$A:$K.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [sheet] Hospital CDM                             → 0 match(es)

[2024] THE GENERAL HOSPITAL
  Processing: 106301340_CDM_ALL.xlsx
  [sheet] AB 1045 Form                             → 4 match(es)
  [sheet] Non-Pharmacy CDM                         → 9 match(es)
  [sheet] Pharmacy CDM                             → 0 match(es)
  [sheet] Supply CDM                               → 0 match(es)
  [sheet] Removed Items                            → 0 match(es)
  [sheet] Calc & Attestation                       → 0 match(es)

[2024] THOUSAND OAKS SURGICAL HOSPITAL,
  Processing: 106560492_25-Common-Optional-Reporting-Form 2024-1.xlsx
  [sheet] Top 50 List                              → 0 match(es)
  [sheet] AB 1045 Form                             → 4 match(es)
  Processing: 106560492_CDM_Jun2024.xlsx
  [sheet] CMD FILE                                 → 0 match(es)
  Processing: 106560492_PCT_CHG _10012024.xlsx
  [sheet] applies to 3 facilities                  → 0 match(es)

[2024] TORRANCE MEM

In [2]:
"""
validate_extraction.py
-----------------------
Validates the chargemaster extraction results by comparing:
1. How many year folders exist in the data
2. How many hospital folders exist per year
3. How many hospitals we actually got data for
4. What CPT/HCPCS codes were found and how many times
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# PATHS — update if needed
# ─────────────────────────────────────────────
BASE_DIR   = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"


# ─────────────────────────────────────────────
# STEP 1 — What exists in the folder structure?
# ─────────────────────────────────────────────
def scan_folder_structure(base_dir):
    print("\n" + "="*60)
    print("STEP 1: FOLDER STRUCTURE SUMMARY")
    print("="*60)

    year_summary = {}

    for entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not entry.is_dir():
            continue
        year_match = re.search(r'(20\d{2})', entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Drill down to find hospital folders
        hospital_dir = entry.path
        for _ in range(3):  # max 3 levels deep
            subdirs = [s for s in os.scandir(hospital_dir) if s.is_dir()]
            if not subdirs:
                break
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f)) for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                break
            hospital_dir = subdirs[0].path

        hospitals = [s.name for s in os.scandir(hospital_dir) if s.is_dir()]
        year_summary[year] = {
            "folder": entry.name,
            "hospital_count": len(hospitals),
            "hospitals": sorted(hospitals)
        }

        print(f"\n  [{year}] Folder: {entry.name}")
        print(f"         Hospitals found: {len(hospitals)}")

    print(f"\n  TOTAL YEARS IN FOLDER : {len(year_summary)}")
    print(f"  YEARS                 : {sorted(year_summary.keys())}")
    total_hospitals = sum(v['hospital_count'] for v in year_summary.values())
    print(f"  TOTAL HOSPITAL FOLDERS: {total_hospitals}")

    return year_summary


# ─────────────────────────────────────────────
# STEP 2 — What did we actually extract?
# ─────────────────────────────────────────────
def validate_extraction(output_dir, year_summary):
    print("\n" + "="*60)
    print("STEP 2: EXTRACTION RESULTS SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    log_path     = os.path.join(output_dir, "extraction_log.csv")
    error_path   = os.path.join(output_dir, "missing_files_or_errors.csv")

    if not os.path.exists(matched_path):
        print("\n  [ERROR] matched_rows.csv not found. Run the extractor first.")
        return

    df      = pd.read_csv(matched_path, dtype=str)
    def safe_read(path):
        try:
            return pd.read_csv(path, dtype=str)
        except Exception:
            return pd.DataFrame()

    log_df  = safe_read(log_path)
    err_df  = safe_read(error_path)

    print(f"\n  Total matched rows     : {len(df)}")
    print(f"  Total sheets processed : {len(log_df)}")
    print(f"  Total files with errors: {len(err_df)}")

    # Years extracted
    years_extracted = sorted(df["year"].dropna().unique())
    print(f"\n  Years with data        : {len(years_extracted)}")
    print(f"  Years                  : {years_extracted}")

    # Hospitals extracted per year
    print(f"\n  {'YEAR':<8} {'HOSPITALS IN FOLDER':>22} {'HOSPITALS WITH DATA':>22} {'TOTAL ROWS':>12}")
    print(f"  {'-'*68}")
    for year in sorted(df["year"].dropna().unique()):
        year_df     = df[df["year"] == year]
        got_data    = year_df["hospital"].nunique()
        in_folder   = year_summary.get(year, {}).get("hospital_count", "N/A")
        rows        = len(year_df)
        print(f"  {year:<8} {str(in_folder):>22} {got_data:>22} {rows:>12}")

    # Missing hospitals (in folder but not in output)
    print(f"\n" + "="*60)
    print("STEP 3: HOSPITALS IN FOLDER BUT NO DATA EXTRACTED")
    print("="*60)
    for year, info in sorted(year_summary.items()):
        if year not in years_extracted:
            print(f"\n  [{year}] — entire year was skipped (not in TEST_YEARS)")
            continue
        year_df         = df[df["year"] == year]
        extracted_hosp  = set(year_df["hospital"].dropna().str.strip().str.upper())
        folder_hosp     = set(h.upper() for h in info["hospitals"])
        missing         = folder_hosp - extracted_hosp
        if missing:
            print(f"\n  [{year}] {len(missing)} hospitals had no matches:")
            for h in sorted(missing)[:10]:  # show max 10
                print(f"    - {h}")
            if len(missing) > 10:
                print(f"    ... and {len(missing)-10} more")
        else:
            print(f"\n  [{year}] All hospitals had at least one match ✓")


# ─────────────────────────────────────────────
# STEP 3 — CPT / HCPCS code summary
# ─────────────────────────────────────────────
def validate_codes(output_dir):
    print(f"\n" + "="*60)
    print("STEP 4: CPT / HCPCS CODE SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    df = pd.read_csv(matched_path, dtype=str)

    # Code type split
    print(f"\n  Code type breakdown:")
    print(df["code_type"].value_counts().to_string())

    # All unique codes found
    code_counts = df.groupby(["procedure_code", "code_type"]).size().reset_index(name="occurrences")
    code_counts = code_counts.sort_values("occurrences", ascending=False)

    print(f"\n  {'CODE':<12} {'TYPE':<8} {'OCCURRENCES':>12}")
    print(f"  {'-'*35}")
    for _, row in code_counts.iterrows():
        print(f"  {row['procedure_code']:<12} {row['code_type']:<8} {row['occurrences']:>12}")

    print(f"\n  TOTAL UNIQUE CODES FOUND: {len(code_counts)}")

    # Codes in target list but NOT found anywhere
    TARGET_CODES = {
        "99281", "99282", "99283", "99284", "99285",  # ER visit levels 1-5
    "99291", "99292",                              # Critical care

    # ── Gynecology ──────────────────────────────────────────────────
    "59400", "59409", "59410",                     # Vaginal delivery
    "59510", "59514", "59515", 
    }
    found_codes  = set(df["procedure_code"].dropna().unique())
    missing_codes = TARGET_CODES - found_codes
    print(f"\n  Target codes NOT found in any file ({len(missing_codes)}):")
    for c in sorted(missing_codes):
        print(f"    - {c}")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("\n" + "="*60)
    print("   CHARGEMASTER EXTRACTION VALIDATOR")
    print("="*60)

    year_summary = scan_folder_structure(BASE_DIR)
    validate_extraction(OUTPUT_DIR, year_summary)
    validate_codes(OUTPUT_DIR)

    print("\n" + "="*60)
    print("   VALIDATION COMPLETE")
    print("="*60)

if __name__ == "__main__":
    main()


   CHARGEMASTER EXTRACTION VALIDATOR

STEP 1: FOLDER STRUCTURE SUMMARY

  [2013] Folder: 2013-hospital-chargemasters
         Hospitals found: 408

  [2014] Folder: 2014-hospital-chargemasters
         Hospitals found: 408

  [2015] Folder: 2015-hospital-chargemasters
         Hospitals found: 404

  [2016] Folder: 2016-hospital-chargemasters
         Hospitals found: 399

  [2017] Folder: 2017-hospital-chargemasters
         Hospitals found: 310

  [2018] Folder: 2018-hospital-chargemasters
         Hospitals found: 1

  [2019] Folder: 2019-hospital-chargemasters
         Hospitals found: 363

  [2023] Folder: 2023-hospital-chargemaster
         Hospitals found: 448

  [2020] Folder: chargemaster-cdm-2020r
         Hospitals found: 334

  [2021] Folder: chargemaster-cdm-2021
         Hospitals found: 318

  [2022] Folder: chargemastercdm-2022
         Hospitals found: 450

  [2024] Folder: chargemastercdm-2024
         Hospitals found: 457

  TOTAL YEARS IN FOLDER : 12
  YEARS       

In [3]:
import pandas as pd
df = pd.read_csv(r"C:\Users\deeksha\OneDrive - Indiana University\californis\output\matched_rows.csv")

# Pick 5 random rows
sample = df.sample(5, random_state=42)
print(sample[["year", "hospital", "file_name", "sheet_name", "procedure_code", "description", "charge"]])

      year                                           hospital  \
1176  2024  FOUNTAIN VALLEY REGIONAL HOSPITAL AND MEDICAL ...   
3362  2024       LOS ANGELES COMMUNITY HOSPITAL AT BELLFLOWER   
1090  2024            EMANATE HEALTH INTER-COMMUNITY HOSPITAL   
4816  2024                 SURPRISE VALLEY COMMUNITY HOSPITAL   
4374  2024                       RIDGECREST REGIONAL HOSPITAL   

                                            file_name  \
1176                           106301175_CDM_All.xlsx   
3362  106190066_CommonProcedures_2024 BELLFLOWER.xlsx   
1090                           106190413_CDM_ALL.xlsx   
4816                           106250955_CDM_ALL.xlsm   
4374                            106150782_CDM_CHG.xls   

                          sheet_name  procedure_code  \
1176               106301175_PCT_CHG           99285   
3362                    AB 1045 Form           99282   
1090                Non-Pharmacy CDM           99285   
4816           106250955 ITEM MASTER      

In [4]:
#1. Manual Spot Check (most important)
import pandas as pd
df = pd.read_csv(r"C:\Users\deeksha\OneDrive - Indiana University\californis\output\matched_rows.csv")

# Pick 5 random rows
sample = df.sample(5, random_state=42)
print(sample[["year", "hospital", "file_name", "sheet_name", "procedure_code", "description", "charge"]])

      year                                           hospital  \
1176  2024  FOUNTAIN VALLEY REGIONAL HOSPITAL AND MEDICAL ...   
3362  2024       LOS ANGELES COMMUNITY HOSPITAL AT BELLFLOWER   
1090  2024            EMANATE HEALTH INTER-COMMUNITY HOSPITAL   
4816  2024                 SURPRISE VALLEY COMMUNITY HOSPITAL   
4374  2024                       RIDGECREST REGIONAL HOSPITAL   

                                            file_name  \
1176                           106301175_CDM_All.xlsx   
3362  106190066_CommonProcedures_2024 BELLFLOWER.xlsx   
1090                           106190413_CDM_ALL.xlsx   
4816                           106250955_CDM_ALL.xlsm   
4374                            106150782_CDM_CHG.xls   

                          sheet_name  procedure_code  \
1176               106301175_PCT_CHG           99285   
3362                    AB 1045 Form           99282   
1090                Non-Pharmacy CDM           99285   
4816           106250955 ITEM MASTER      

In [5]:
print(f"Rows with charge      : {df['charge_numeric'].notna().sum()}")
print(f"Rows WITHOUT charge   : {df['charge_numeric'].isna().sum()}")
print(f"% rows missing charge : {df['charge_numeric'].isna().mean()*100:.1f}%")

KeyError: 'charge_numeric'

In [6]:
import pandas as pd

# Load the data
df = pd.read_csv(r"C:\Users\deeksha\OneDrive - Indiana University\californis\output\matched_rows.csv")

# Create charge_numeric column
df["charge_numeric"] = pd.to_numeric(df["charge"], errors="coerce")

# Create code_valid column
import re
CPT_PATTERN   = re.compile(r"^\d{5}([A-Z0-9]{2})?$")
HCPCS_PATTERN = re.compile(r"^[A-Z]\d{4}$")
df["code_valid"] = df["procedure_code"].apply(
    lambda x: bool(CPT_PATTERN.match(str(x)) or HCPCS_PATTERN.match(str(x)))
)

# Now run the checks
print("="*50)
print("DATA QUALITY SUMMARY")
print("="*50)
print(f"Total rows              : {len(df)}")
print(f"Years covered           : {sorted(df['year'].unique())}")
print(f"Unique hospitals        : {df['hospital'].nunique()}")
print(f"Unique CPT/HCPCS codes  : {df['procedure_code'].nunique()}")
print(f"Rows with valid charge  : {df['charge_numeric'].notna().sum()}")
print(f"Rows missing charge     : {df['charge_numeric'].isna().sum()}")
print(f"% rows missing charge   : {df['charge_numeric'].isna().mean()*100:.1f}%")
print(f"Duplicate rows          : {df.duplicated().sum()}")
print(f"Invalid codes           : {(~df['code_valid']).sum()}")
print(f"Suspicious charges >1M  : {(df['charge_numeric'] > 1000000).sum()}")

DATA QUALITY SUMMARY
Total rows              : 5149
Years covered           : [np.int64(2024)]
Unique hospitals        : 391
Unique CPT/HCPCS codes  : 13
Rows with valid charge  : 4251
Rows missing charge     : 898
% rows missing charge   : 17.4%
Duplicate rows          : 151
Invalid codes           : 0
Suspicious charges >1M  : 0


In [7]:
"""
chargemaster_extractor.py
--------------------------
Recursively traverses a folder structure organized by year → hospital → files,
extracts ALL rows containing valid CPT or HCPCS codes, and writes three output CSVs.
No target list required — every valid code found is captured.

Expected layout:
    base_dir/
        2013/
            HospitalA/
                chargemaster.xlsx
            HospitalB/
                cdm.csv
        2014/
            ...

Outputs (written next to this script):
    matched_rows.csv        – all matched rows with metadata
    extraction_log.csv      – per-sheet processing summary
    missing_files_or_errors.csv – files that could not be read
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# Root directory containing year folders
BASE_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"

# Where to write the three output files
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"

# ─────────────────────────────────────────────
# TARGET CPT / HCPCS CODES
# ─────────────────────────────────────────────
# Emergency Medicine CPT codes (99281-99285 = ER visits by severity)
# Gynecology CPT codes (common OB/GYN procedures)
# Add or remove codes here as needed for your research.
TARGET_CODES = {
    # ── Emergency Medicine ──────────────────────────────────────────
    "99281", "99282", "99283", "99284", "99285",  # ER visit levels 1-5
    "99291", "99292",                              # Critical care

    # ── Gynecology ──────────────────────────────────────────────────
    "59400", "59409", "59410",                     # Vaginal delivery
    "59510", "59514", "59515",                     # Cesarean delivery
}

# ─────────────────────────────────────────────
# REGEX PATTERNS
# ─────────────────────────────────────────────

# Column-name patterns that likely contain procedure codes
CODE_COL_PATTERNS = re.compile(
    r"(cpt|hcpcs|procedure\s*code|proc\s*code|cpt[\s/\-]*hcpcs|service\s*code|"
    r"\bcode\b|billing\s*code|revenue\s*code)",
    re.IGNORECASE,
)

# Column-name patterns for description and charge
DESC_COL_PATTERNS = re.compile(
    r"(description|desc|procedure\s*name|service\s*name|item\s*name|narrative)",
    re.IGNORECASE,
)
CHARGE_COL_PATTERNS = re.compile(
    # Match price/amount columns but EXCLUDE "Charge Code" (internal hospital ID)
    # Also catches "June 2024 Prices", "Average Charge", "Gross Charge" etc.
    r"(?i)^(?!charge\s*code)(?=.*(price|amount|rate|average\s*charge|avg\s*charge|"
    r"gross\s*charge|standard\s*charge|billed\s*charge|cdm\s*price|list\s*price|"
    r"\d{4}\s*price))",
    re.IGNORECASE,
)

# CPT codes: 5 digits (optionally followed by a 2-char modifier)
CPT_PATTERN = re.compile(r"^\d{5}([A-Z0-9]{2})?$")

# HCPCS Level II codes: letter + 4 digits
HCPCS_PATTERN = re.compile(r"^[A-Z]\d{4}$")


# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────


def normalize_code(value) -> str:
    """Strip whitespace, uppercase, remove non-alphanumeric characters."""
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())


def detect_code_type(code: str) -> str:
    """Return 'CPT', 'HCPCS', or 'UNKNOWN' based on code format."""
    if CPT_PATTERN.match(code):
        return "CPT"
    if HCPCS_PATTERN.match(code):
        return "HCPCS"
    return "UNKNOWN"


def find_columns(df: pd.DataFrame, pattern: re.Pattern) -> list:
    """Return column names matching a regex pattern."""
    return [c for c in df.columns if pattern.search(str(c))]


# Keywords that strongly indicate a real header row
HEADER_KEYWORDS = re.compile(
    r"(cpt|hcpcs|code|charge|price|amount|description|desc|procedure|service|rate|cost|revenue)",
    re.IGNORECASE,
)

def find_header_row(df_raw: pd.DataFrame, max_scan: int = 15) -> int:
    """
    Find the real header row by scoring each row on two criteria:
    1. KEYWORD score  — how many cells contain header-like words (cpt, charge, description etc.)
    2. BREADTH score  — how many non-empty string cells (width of the header)
    Keyword hits are weighted heavily so title rows like "Common OP Procedures"
    don't outscore a real multi-column header.
    """
    best_row, best_score = 0, -1
    for i in range(min(max_scan, len(df_raw))):
        row = df_raw.iloc[i]
        keyword_hits = sum(
            1 for v in row
            if isinstance(v, str) and HEADER_KEYWORDS.search(v)
        )
        breadth = sum(
            1 for v in row
            if isinstance(v, str) and len(v.strip()) > 1
        )
        # Keyword hits worth 5x more than plain text breadth
        score = keyword_hits * 5 + breadth
        if score > best_score:
            best_score, best_row = score, i
    return best_row


def get_engine(filepath: str) -> str:
    """Return the correct pandas engine based on file extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".xls":
        return "xlrd"
    return "openpyxl"


def read_sheet(filepath: str, sheet_name, engine: str) -> pd.DataFrame:
    """
    Read a single Excel sheet in one pass, auto-detecting the header row.
    Uses xlrd for .xls and openpyxl for .xlsx/.xlsm.
    """
    # Single raw read to find header row and return data
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None,
                        dtype=str, engine=engine)
    header_row = find_header_row(raw)
    # Slice: rows above header become the header, rows below are data
    df = raw.iloc[header_row + 1:].copy()
    df.columns = [str(v).strip() for v in raw.iloc[header_row]]
    df.reset_index(drop=True, inplace=True)
    return df


def read_csv_file(filepath: str) -> pd.DataFrame:
    """Read a CSV, trying common encodings."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(filepath, dtype=str, encoding=enc)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode CSV: {filepath}")


# ─────────────────────────────────────────────
# CORE EXTRACTION LOGIC
# ─────────────────────────────────────────────

def is_valid_code(code: str) -> bool:
    """Return True if code is a valid CPT/HCPCS format AND in our target list."""
    is_valid_format = bool(CPT_PATTERN.match(code) or HCPCS_PATTERN.match(code))
    return is_valid_format and code in TARGET_CODES


def extract_matches(
    df: pd.DataFrame,
    year: str,
    hospital: str,
    filename: str,
    sheet_name: str,
) -> tuple[list[dict], dict]:
    """
    Scan `df` and extract ALL rows that contain a valid CPT or HCPCS code.

    Returns:
        matched_rows  – list of dicts (one per matched row)
        log_entry     – dict summarising what was detected in this sheet
    """
    # Identify candidate code columns by name
    code_cols = find_columns(df, CODE_COL_PATTERNS)
    desc_cols = find_columns(df, DESC_COL_PATTERNS)
    charge_cols = find_columns(df, CHARGE_COL_PATTERNS)

    # If no named code column found, scan ALL columns for target code values
    if not code_cols:
        for col in df.columns:
            sample = df[col].dropna().head(100)
            hits = sample.apply(lambda v: is_valid_code(normalize_code(v))).sum()
            if hits >= 1:  # even 1 target code hit is enough
                code_cols.append(col)

    matched_rows = []

    for code_col in code_cols:
        for _, row in df.iterrows():
            raw_val = row.get(code_col, "")
            norm = normalize_code(raw_val)
            if not is_valid_code(norm):
                continue  # skip blanks, internal codes, non-CPT/HCPCS values

            description = next(
                (row[c] for c in desc_cols if pd.notna(row.get(c))), ""
            )
            charge = next(
                (row[c] for c in charge_cols if pd.notna(row.get(c))), ""
            )

            matched_rows.append({
                "year":          year,
                "hospital":      hospital,
                "file_name":     filename,
                "sheet_name":    sheet_name,
                "source_column": code_col,
                "procedure_code": norm,
                "code_type":     detect_code_type(norm),
                "description":   str(description).strip(),
                "charge":        str(charge).strip(),
            })

    log_entry = {
        "file_name":         filename,
        "sheet_name":        sheet_name,
        "code_cols_found":   ", ".join(code_cols) if code_cols else "NONE",
        "desc_cols_found":   ", ".join(desc_cols) if desc_cols else "NONE",
        "charge_cols_found": ", ".join(charge_cols) if charge_cols else "NONE",
        "num_matches":       len(matched_rows),
    }
    return matched_rows, log_entry


# ─────────────────────────────────────────────
# FILE PROCESSING
# ─────────────────────────────────────────────

def process_file(
    filepath: str,
    year: str,
    hospital: str,
) -> tuple[list[dict], list[dict], list[dict]]:
    """
    Process one file (Excel or CSV).

    Returns:
        all_matches   – matched rows from this file
        log_entries   – one log entry per sheet/file
        error_entries – populated only on failure
    """
    filename = os.path.basename(filepath)
    all_matches, log_entries, error_entries = [], [], []

    try:
        ext = os.path.splitext(filepath)[1].lower()

        if ext in (".xlsx", ".xls", ".xlsm"):
            engine = get_engine(filepath)
            xl = pd.ExcelFile(filepath, engine=engine)
            sheets = xl.sheet_names
            for sheet in sheets:
                try:
                    df = read_sheet(filepath, sheet, engine)
                    matches, log = extract_matches(
                        df, year, hospital, filename, sheet
                    )
                    all_matches.extend(matches)
                    log_entries.append(log)
                    print(
                        f"  [sheet] {sheet:40s} → {log['num_matches']} match(es)"
                    )
                except Exception as sheet_err:
                    print(f"  [WARN] Could not read sheet '{sheet}': {sheet_err}")
                    log_entries.append({
                        "file_name": filename,
                        "sheet_name": sheet,
                        "code_cols_found": "ERROR",
                        "desc_cols_found": "",
                        "charge_cols_found": "",
                        "num_matches": 0,
                    })

        elif ext == ".csv":
            df = read_csv_file(filepath)
            matches, log = extract_matches(
                df, year, hospital, filename, "N/A"
            )
            all_matches.extend(matches)
            log_entries.append(log)
            print(f"  [csv ] {filename:40s} → {log['num_matches']} match(es)")

        else:
            print(f"  [SKIP] Unsupported file type: {filename}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {filename}: {e}")
        error_entries.append({
            "file_path": filepath,
            "year":      year,
            "hospital":  hospital,
            "error":     str(e),
        })

    return all_matches, log_entries, error_entries


# ─────────────────────────────────────────────
# DIRECTORY TRAVERSAL
# ─────────────────────────────────────────────

def traverse_and_extract(base_dir: str):
    """
    Walk base_dir/year/hospital/files and collect all results.
    Folder depth is: base_dir → year → hospital → (any depth of files).
    """
    all_matches, all_logs, all_errors = [], [], []

    if not os.path.isdir(base_dir):
        print(f"[ERROR] Base directory not found: {base_dir}")
        return all_matches, all_logs, all_errors

    # ── DEMO MODE SETTINGS ───────────────────────────────────────────────
    # Hospitals to sample per year for demo (set to None for all hospitals)
    DEMO_HOSPITALS_PER_YEAR = 3  # None = ALL hospitals, set a number e.g. 3 to limit
    # Only process 2018 and after
    # None = ALL years | customize as needed:
    #   TEST_YEARS = {"2018"}               → only 2018
    #   TEST_YEARS = {"2018", "2019"}        → custom years
    #   TEST_YEARS = None                    → all years
    TEST_YEARS = {"2024"}  # only 2024 for testing
    # ─────────────────────────────────────────────────────────────────────

    for year_entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not year_entry.is_dir():
            continue
        # Extract 4-digit year from folder name regardless of naming convention
        # Handles: '2014-hospital-chargemasters', 'chargemaster-cdm-2020r', etc.
        year_match = re.search(r'(20\d{2})', year_entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Skip years outside test range
        if TEST_YEARS and year not in TEST_YEARS:
            print(f"[SKIP] {year_entry.name}")
            continue

        # Drill down through subfolders until we find hospital folders
        # Handles: year/hospitals, year/year/hospitals, year/folder/hospitals
        def find_hospital_dir(path, depth=0):
            if depth > 3:
                return path
            subdirs = [s for s in os.scandir(path) if s.is_dir()]
            if not subdirs:
                return path
            # If subdirs contain files -> these are hospital folders
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f))
                    for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                return path
            # Otherwise go deeper
            return find_hospital_dir(subdirs[0].path, depth + 1)

        hospital_search_dir = find_hospital_dir(year_entry.path)
        print(f"[{year}] Hospital dir: {os.path.relpath(hospital_search_dir, year_entry.path)}")


        all_hospitals = sorted(
            [e for e in os.scandir(hospital_search_dir) if e.is_dir()],
            key=lambda e: e.name
        )
        # In demo mode, only take first N hospitals per year
        if DEMO_HOSPITALS_PER_YEAR:
            all_hospitals = all_hospitals[:DEMO_HOSPITALS_PER_YEAR]
            print(f"  [DEMO] Sampling {len(all_hospitals)} hospitals for {year}")

        for hospital_entry in all_hospitals:
            hospital = hospital_entry.name
            print(f"\n[{year}] {hospital}")

            # Only look at files directly inside the hospital folder (no deeper)
            # os.walk was recursing into sibling folders causing hospital mismatch
            for fname in os.listdir(hospital_entry.path):
                ext = os.path.splitext(fname)[1].lower()
                if ext not in (".xlsx", ".xls", ".xlsm", ".csv"):
                    continue
                fpath = os.path.join(hospital_entry.path, fname)
                if not os.path.isfile(fpath):
                    continue
                print(f"  Processing: {fname}")
                matches, logs, errors = process_file(
                    fpath, year, hospital
                )
                all_matches.extend(matches)
                all_logs.extend(logs)
                all_errors.extend(errors)

    return all_matches, all_logs, all_errors


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Traverse directory and extract all CPT/HCPCS codes
    print(f"\n[INFO] Scanning: {BASE_DIR}\n{'─'*60}")
    all_matches, all_logs, all_errors = traverse_and_extract(BASE_DIR)

    # 3. Save matched_rows.csv
    matched_path = os.path.join(OUTPUT_DIR, "matched_rows.csv")
    matched_cols = [
        "year", "hospital", "file_name", "sheet_name",
        "source_column", "procedure_code", "code_type",
        "description", "charge",
    ]
    if all_matches:
        pd.DataFrame(all_matches, columns=matched_cols).to_csv(
            matched_path, index=False
        )
    else:
        pd.DataFrame(columns=matched_cols).to_csv(matched_path, index=False)
    print(f"\n[OUT] matched_rows.csv          → {len(all_matches)} row(s)")

    # 4. Save extraction_log.csv
    log_path = os.path.join(OUTPUT_DIR, "extraction_log.csv")
    pd.DataFrame(all_logs).to_csv(log_path, index=False)
    print(f"[OUT] extraction_log.csv        → {len(all_logs)} sheet(s) processed")

    # 5. Save missing_files_or_errors.csv
    err_path = os.path.join(OUTPUT_DIR, "missing_files_or_errors.csv")
    pd.DataFrame(all_errors).to_csv(err_path, index=False)
    print(f"[OUT] missing_files_or_errors.csv → {len(all_errors)} error(s)")

    print("\n[DONE] Extraction complete.")


if __name__ == "__main__":
    main()


[INFO] Scanning: C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped
────────────────────────────────────────────────────────────
[SKIP] 2013-hospital-chargemasters
[SKIP] 2014-hospital-chargemasters
[SKIP] 2015-hospital-chargemasters
[SKIP] 2016-hospital-chargemasters
[SKIP] 2017-hospital-chargemasters
[SKIP] 2018-hospital-chargemasters
[SKIP] 2019-hospital-chargemasters
[SKIP] 2023-hospital-chargemaster
[SKIP] chargemaster-cdm-2020r
[SKIP] chargemaster-cdm-2021
[SKIP] chargemastercdm-2022
[2024] Hospital dir: ChargemasterCDM-2024
  [DEMO] Sampling 3 hospitals for 2024

[2024] ADVENTIST HEALTH AND RIDEOUT
  Processing: 106580996_CDM_All_AHRO_2024.xlsx
  [sheet] Cover Page                               → 0 match(es)
  [sheet] Price Change Impact                      → 0 match(es)
  [sheet] Common OP Procedures                     → 4 match(es)
  [sheet] Hospital CDM                             → 31 match(es)

[2024] ADVENTIST HEALTH BAKERSFIELD
  Processing: 106150

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\deeksha\\OneDrive - Indiana University\\californis\\output\\matched_rows.csv'

In [8]:
"""
validate_extraction.py
-----------------------
Validates the chargemaster extraction results by comparing:
1. How many year folders exist in the data
2. How many hospital folders exist per year
3. How many hospitals we actually got data for
4. What CPT/HCPCS codes were found and how many times
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────────
# PATHS — update if needed
# ─────────────────────────────────────────────
BASE_DIR   = r"C:\Users\deeksha\OneDrive - Indiana University\californis\data_unzipped"
OUTPUT_DIR = r"C:\Users\deeksha\OneDrive - Indiana University\californis\output"


# ─────────────────────────────────────────────
# STEP 1 — What exists in the folder structure?
# ─────────────────────────────────────────────
def scan_folder_structure(base_dir):
    print("\n" + "="*60)
    print("STEP 1: FOLDER STRUCTURE SUMMARY")
    print("="*60)

    year_summary = {}

    for entry in sorted(os.scandir(base_dir), key=lambda e: e.name):
        if not entry.is_dir():
            continue
        year_match = re.search(r'(20\d{2})', entry.name)
        if not year_match:
            continue
        year = year_match.group(1)

        # Drill down to find hospital folders
        hospital_dir = entry.path
        for _ in range(3):  # max 3 levels deep
            subdirs = [s for s in os.scandir(hospital_dir) if s.is_dir()]
            if not subdirs:
                break
            has_files = any(
                any(os.path.isfile(os.path.join(s.path, f)) for f in os.listdir(s.path))
                for s in subdirs
            )
            if has_files:
                break
            hospital_dir = subdirs[0].path

        hospitals = [s.name for s in os.scandir(hospital_dir) if s.is_dir()]
        year_summary[year] = {
            "folder": entry.name,
            "hospital_count": len(hospitals),
            "hospitals": sorted(hospitals)
        }

        print(f"\n  [{year}] Folder: {entry.name}")
        print(f"         Hospitals found: {len(hospitals)}")

    print(f"\n  TOTAL YEARS IN FOLDER : {len(year_summary)}")
    print(f"  YEARS                 : {sorted(year_summary.keys())}")
    total_hospitals = sum(v['hospital_count'] for v in year_summary.values())
    print(f"  TOTAL HOSPITAL FOLDERS: {total_hospitals}")

    return year_summary


# ─────────────────────────────────────────────
# STEP 2 — What did we actually extract?
# ─────────────────────────────────────────────
def validate_extraction(output_dir, year_summary):
    print("\n" + "="*60)
    print("STEP 2: EXTRACTION RESULTS SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    log_path     = os.path.join(output_dir, "extraction_log.csv")
    error_path   = os.path.join(output_dir, "missing_files_or_errors.csv")

    if not os.path.exists(matched_path):
        print("\n  [ERROR] matched_rows.csv not found. Run the extractor first.")
        return

    df      = pd.read_csv(matched_path, dtype=str)
    def safe_read(path):
        try:
            return pd.read_csv(path, dtype=str)
        except Exception:
            return pd.DataFrame()

    log_df  = safe_read(log_path)
    err_df  = safe_read(error_path)

    print(f"\n  Total matched rows     : {len(df)}")
    print(f"  Total sheets processed : {len(log_df)}")
    print(f"  Total files with errors: {len(err_df)}")

    # Years extracted
    years_extracted = sorted(df["year"].dropna().unique())
    print(f"\n  Years with data        : {len(years_extracted)}")
    print(f"  Years                  : {years_extracted}")

    # Hospitals extracted per year
    print(f"\n  {'YEAR':<8} {'HOSPITALS IN FOLDER':>22} {'HOSPITALS WITH DATA':>22} {'TOTAL ROWS':>12}")
    print(f"  {'-'*68}")
    for year in sorted(df["year"].dropna().unique()):
        year_df     = df[df["year"] == year]
        got_data    = year_df["hospital"].nunique()
        in_folder   = year_summary.get(year, {}).get("hospital_count", "N/A")
        rows        = len(year_df)
        print(f"  {year:<8} {str(in_folder):>22} {got_data:>22} {rows:>12}")

    # Missing hospitals (in folder but not in output)
    print(f"\n" + "="*60)
    print("STEP 3: HOSPITALS IN FOLDER BUT NO DATA EXTRACTED")
    print("="*60)
    for year, info in sorted(year_summary.items()):
        if year not in years_extracted:
            print(f"\n  [{year}] — entire year was skipped (not in TEST_YEARS)")
            continue
        year_df         = df[df["year"] == year]
        extracted_hosp  = set(year_df["hospital"].dropna().str.strip().str.upper())
        folder_hosp     = set(h.upper() for h in info["hospitals"])
        missing         = folder_hosp - extracted_hosp
        if missing:
            print(f"\n  [{year}] {len(missing)} hospitals had no matches:")
            for h in sorted(missing)[:10]:  # show max 10
                print(f"    - {h}")
            if len(missing) > 10:
                print(f"    ... and {len(missing)-10} more")
        else:
            print(f"\n  [{year}] All hospitals had at least one match ✓")


# ─────────────────────────────────────────────
# STEP 3 — CPT / HCPCS code summary
# ─────────────────────────────────────────────
def validate_codes(output_dir):
    print(f"\n" + "="*60)
    print("STEP 4: CPT / HCPCS CODE SUMMARY")
    print("="*60)

    matched_path = os.path.join(output_dir, "matched_rows.csv")
    df = pd.read_csv(matched_path, dtype=str)

    # Code type split
    print(f"\n  Code type breakdown:")
    print(df["code_type"].value_counts().to_string())

    # All unique codes found
    code_counts = df.groupby(["procedure_code", "code_type"]).size().reset_index(name="occurrences")
    code_counts = code_counts.sort_values("occurrences", ascending=False)

    print(f"\n  {'CODE':<12} {'TYPE':<8} {'OCCURRENCES':>12}")
    print(f"  {'-'*35}")
    for _, row in code_counts.iterrows():
        print(f"  {row['procedure_code']:<12} {row['code_type']:<8} {row['occurrences']:>12}")

    print(f"\n  TOTAL UNIQUE CODES FOUND: {len(code_counts)}")

    # Codes in target list but NOT found anywhere
    TARGET_CODES = {
        # ── Emergency Medicine ──────────────────────────────────────
        "99281", "99282", "99283", "99284", "99285",  # ER visit levels 1-5
        "99291", "99292",                              # Critical care

        # ── Gynecology ──────────────────────────────────────────────
        "59400", "59409", "59410",                     # Vaginal delivery
        "59510", "59514", "59515",                     # Cesarean delivery
    }
    found_codes  = set(df["procedure_code"].dropna().unique())
    missing_codes = TARGET_CODES - found_codes
    print(f"\n  Target codes NOT found in any file ({len(missing_codes)}):")
    for c in sorted(missing_codes):
        print(f"    - {c}")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("\n" + "="*60)
    print("   CHARGEMASTER EXTRACTION VALIDATOR")
    print("="*60)

    year_summary = scan_folder_structure(BASE_DIR)
    validate_extraction(OUTPUT_DIR, year_summary)
    validate_codes(OUTPUT_DIR)

    print("\n" + "="*60)
    print("   VALIDATION COMPLETE")
    print("="*60)

if __name__ == "__main__":
    main()


   CHARGEMASTER EXTRACTION VALIDATOR

STEP 1: FOLDER STRUCTURE SUMMARY

  [2013] Folder: 2013-hospital-chargemasters
         Hospitals found: 408

  [2014] Folder: 2014-hospital-chargemasters
         Hospitals found: 408

  [2015] Folder: 2015-hospital-chargemasters
         Hospitals found: 404

  [2016] Folder: 2016-hospital-chargemasters
         Hospitals found: 399

  [2017] Folder: 2017-hospital-chargemasters
         Hospitals found: 310

  [2018] Folder: 2018-hospital-chargemasters
         Hospitals found: 1

  [2019] Folder: 2019-hospital-chargemasters
         Hospitals found: 363

  [2023] Folder: 2023-hospital-chargemaster
         Hospitals found: 448

  [2020] Folder: chargemaster-cdm-2020r
         Hospitals found: 334

  [2021] Folder: chargemaster-cdm-2021
         Hospitals found: 318

  [2022] Folder: chargemastercdm-2022
         Hospitals found: 450

  [2024] Folder: chargemastercdm-2024
         Hospitals found: 457

  TOTAL YEARS IN FOLDER : 12
  YEARS       

In [9]:
import pandas as pd

df = pd.read_csv(r"C:\Users\deeksha\OneDrive - Indiana University\californis\output\matched_rows.csv")
df["charge_numeric"] = pd.to_numeric(df["charge"], errors="coerce")

# 1. Basic price stats per CPT code
print("=== PRICE STATS PER CPT CODE ===")
print(df.groupby("procedure_code")["charge_numeric"].describe().round(2))

# 2. Sample rows WITH charges
print("\n=== 5 ROWS WITH VALID CHARGES ===")
has_charge = df[df["charge_numeric"].notna()]
print(has_charge[["hospital","procedure_code","description","charge"]].sample(5))

# 3. Sample rows WITHOUT charges
print("\n=== 5 ROWS MISSING CHARGES ===")
no_charge = df[df["charge_numeric"].isna()]
print(no_charge[["hospital","file_name","sheet_name","procedure_code","charge"]].head(5))

# 4. Suspiciously low or high prices
print("\n=== SUSPICIOUS PRICES ===")
print(f"Charges under $10    : {(df['charge_numeric'] < 10).sum()}")
print(f"Charges over $100,000: {(df['charge_numeric'] > 100000).sum()}")
print(f"Charges = 0          : {(df['charge_numeric'] == 0).sum()}")

# 5. Price range per code
print("\n=== MIN/MAX PRICE PER CODE ===")
print(df.groupby("procedure_code")["charge_numeric"].agg(["min","max","median"]).round(2))

=== PRICE STATS PER CPT CODE ===
                count     mean      std     min      25%       50%       75%  \
procedure_code                                                                 
59400            27.0  6339.57  5062.67     0.0  3893.71   4687.00   7926.50   
59409            78.0  5106.87  3801.55     0.0  2597.46   3950.20   6711.78   
59410            11.0  8554.70  6293.01     0.0  1810.36  13800.00  13800.00   
59510            20.0  3290.94  3012.56     0.0  1012.50   1844.26   5889.25   
59514            34.0  3892.30  5024.51     0.0   456.84   1961.69   5504.37   
59515             8.0  1584.46  1224.63  1012.5  1012.50   1012.50   1296.88   
99281           167.0  1055.79  3369.54     0.0   188.50    350.00    608.00   
99282           907.0  1278.61   659.61     0.0  1135.00   1360.00   1360.00   
99283           924.0  2254.59   994.40     0.0  1811.25   2520.00   2520.00   
99284           915.0  3887.56  1647.31     0.0  3000.00   4480.00   4480.00   
99285  

In [10]:
# Check 1 — Are the duplicates actually duplicates or different prices?
dupes = df[df.duplicated(subset=["hospital","procedure_code"], keep=False)]
print(dupes[["hospital","procedure_code","charge","sheet_name","file_name"]]
      .sort_values(["hospital","procedure_code"]).head(20))

# Check 2 — What do the zero charge rows look like?
zeros = df[df["charge_numeric"] == 0]
print(zeros[["hospital","procedure_code","charge","sheet_name"]].head(10))

# Check 3 — What files are PCT_CHG?
pct = df[df["file_name"].str.contains("PCT_CHG|CHG", case=False, na=False)]
print(pct[["hospital","file_name","procedure_code","charge"]].head(10))

                        hospital  procedure_code   charge  \
0   ADVENTIST HEALTH AND RIDEOUT           99282      779   
5   ADVENTIST HEALTH AND RIDEOUT           99282      779   
30  ADVENTIST HEALTH AND RIDEOUT           99282       87   
1   ADVENTIST HEALTH AND RIDEOUT           99283     1059   
7   ADVENTIST HEALTH AND RIDEOUT           99283     1056   
10  ADVENTIST HEALTH AND RIDEOUT           99283  1067.13   
26  ADVENTIST HEALTH AND RIDEOUT           99283     8130   
27  ADVENTIST HEALTH AND RIDEOUT           99283     3956   
33  ADVENTIST HEALTH AND RIDEOUT           99283      148   
2   ADVENTIST HEALTH AND RIDEOUT           99284     2066   
6   ADVENTIST HEALTH AND RIDEOUT           99284     1056   
8   ADVENTIST HEALTH AND RIDEOUT           99284  2014.04   
24  ADVENTIST HEALTH AND RIDEOUT           99284     9088   
25  ADVENTIST HEALTH AND RIDEOUT           99284     4914   
34  ADVENTIST HEALTH AND RIDEOUT           99284      250   
3   ADVENTIST HEALTH AND

In [11]:
import pandas as pd
from pathlib import Path

RAW = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data\raw")
PROC = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data\processed")
PROC.mkdir(parents=True, exist_ok=True)

impact = pd.read_excel(
    RAW / "FY 2026 IPPS Final Rule Impact File.xlsx",
    dtype={"Provider Number": str}
)

print("Shape:", impact.shape)
print("Columns:", impact.columns.tolist())
print(impact.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\deeksha\\OneDrive - Indiana University\\ipps_project\\data\\raw\\FY 2026 IPPS Final Rule Impact File.xlsx'

In [12]:
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

print("Files in data\\:")
for f in BASE.iterdir():
    print(" ", f.name)

raw = BASE / "raw"
if raw.exists():
    print("\nFiles in data\\raw\\:")
    for f in raw.iterdir():
        print(" ", f.name)
else:
    print("\nNo data\\raw\\ folder exists.")

Files in data\:
  CMS-1833-F Table 5.xlsx
  CMS-1833-F Tables 1A - 1E.xlsx
  CMS-1833-F Tables 2, 3, 4A, 4B.xlsx
  cms-1833-f-table-5.zip
  cms-1833-f-tables-1a-1e (1).zip
  cms-1833-f-tables-2-3-4a-4b_0.zip
  FY 2026 IPPS Final Rule Impact File.xlsx
  FY2026_ipps_file_c
  fy_2026_ipps_final_rule_impact_file.zip
  processed

No data\raw\ folder exists.


In [14]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")
PROC = BASE / "processed"

impact = pd.read_excel(
    BASE / "FY 2026 IPPS Final Rule Impact File.xlsx",
    dtype={"Provider Number": str}
)

print("Shape:", impact.shape)
print("Columns:", impact.columns.tolist())
print(impact.head(3))

Shape: (90, 2)
Columns: ['FY 2026 Final Rule IPPS Impact File', 'Unnamed: 1']
                 FY 2026 Final Rule IPPS Impact File  \
0  DATA SOURCES FOR THIS FY 2026 RULE IMPACT FILE...   
1  DISCLAIMER: The variables in the impact file a...   
2                                    Provider Number   

                                          Unnamed: 1  
0                                                NaN  
1                                                NaN  
2  6-digit Medicare provider number; the first 2 ...  


In [15]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

xl = pd.ExcelFile(BASE / "FY 2026 IPPS Final Rule Impact File.xlsx")
print("Sheets:", xl.sheet_names)

Sheets: ['Variable Descriptions', 'FY 2026 FR']


In [16]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

impact = pd.read_excel(
    BASE / "FY 2026 IPPS Final Rule Impact File.xlsx",
    sheet_name="FY 2026 FR",
    dtype={"Provider Number": str}
)

print("Shape:", impact.shape)
print("Columns:", impact.columns.tolist())
print(impact.head(3))

Shape: (3104, 62)
Columns: ['FY 2026 IPPS Impact File -  Final Rule (August 2025)', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61']
  FY 2026 

In [17]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

impact = pd.read_excel(
    BASE / "FY 2026 IPPS Final Rule Impact File.xlsx",
    sheet_name="FY 2026 FR",
    header=1,                              # second row is the real header
    dtype={"Provider Number": str}
)

print("Shape:", impact.shape)
print("Columns:", impact.columns.tolist())
print(impact[["Provider Number", "Name"]].head(3))

Shape: (3103, 62)
Columns: ['Provider Number', 'Name', 'Geographic Labor Market Area', 'Pre-Reclass Labor Market Area', 'Post Reclass Labor Market Area', 'Payment Labor Market Area', 'FIPS County Code', 'Region', 'URGEO', 'URSPA', 'RECLASS', 'FY 2026 Wage Index', 'LUGAR', 'Section 401 Hospital', 'Section 401 hospital with a LUGAR or MGCRB Wage Index Reclass', 'Section 505 Eligible', 'Section 505 Adjustment', 'Cost of Living Adjustment', 'Resident to Bed Ratio', 'RDAY', 'Beds', 'Average Daily Census', 'TCHOP', 'TCHCP', 'DSHPCT', 'DSHOPP', 'DSHCPP', 'DSH_LY', 'UCP_ADJ', 'UCP Per Claim Amount', 'UCP_ADJ_LY', 'UCP Per Claim Amount LY', 'Operating CCR', 'Capital CCR', 'Provider Type', 'HSP Rate', 'Bills', 'CASETA42', 'CMIV42', 'TACMIV42', 'IME_CASETA42', 'IME_TACMIV42', 'CASETA43', 'CMIV43', 'TACMIV43', 'IME_CASETA43', 'IME_TACMIV43', 'GAF', 'Capital Cost of Living Adjustment', 'OUTFACT_F', 'COUTFACT_F', 'Medicare Percentage', 'Medicaid Percentage', 'Proxy Value Based Purchasing Adjustment 

In [18]:
# Filter to California (CCN starts with "05")
ca = impact[impact["Provider Number"].str.startswith("05")].copy()

print("California hospitals in Impact File:", len(ca))
print("\nProvider Type breakdown:")
print(ca["Provider Type"].value_counts(dropna=False))

print("\nFirst 5:")
print(ca[["Provider Number", "Name", "Payment Labor Market Area", "FY 2026 Wage Index"]].head())

California hospitals in Impact File: 278

Provider Type breakdown:
Provider Type
0     212
7      54
16      9
17      3
Name: count, dtype: int64

First 5:
    Provider Number                                               Name  \
191          050002                                   St Rose Hospital   
192          050006                      Providence St Joseph Hospital   
193          050007                     Mills-Peninsula Medical Center   
194          050008  California Pacific Medical Ctr-Davies Campus Hosp   
195          050009      Providence Queen Of The Valley Medical Center   

     Payment Labor Market Area  FY 2026 Wage Index  
191                      36084              1.6529  
192                          5              1.4315  
193                      41884              1.7020  
194                      41884              1.6917  
195                          5              1.6496  


In [19]:
# Keep IPPS-paid hospitals only (exclude SCH for clean Dallas test)
# Provider Type: 0 = STAC, 7 = RRC, 16 = MDH all use standard IPPS payment
# Provider Type 17 = SCH uses hospital-specific rate (different formula)

ca_ipps = ca[ca["Provider Type"].isin([0, 7, 16])].copy()

print("CA IPPS hospitals (eligible for Dallas test):", len(ca_ipps))
print("Excluded (SCH):", (ca["Provider Type"] == 17).sum())

# Quick sanity checks
print("\nMissing wage index:", ca_ipps["FY 2026 Wage Index"].isna().sum())
print("Wage index range:", ca_ipps["FY 2026 Wage Index"].min(), "-", ca_ipps["FY 2026 Wage Index"].max())

print("\nPayment CBSA distribution (top 10):")
print(ca_ipps["Payment Labor Market Area"].value_counts().head(10))

CA IPPS hospitals (eligible for Dallas test): 275
Excluded (SCH): 3

Missing wage index: 0
Wage index range: 1.4315 - 1.7328

Payment CBSA distribution (top 10):
Payment Labor Market Area
5        61
31084    54
40140    22
11244    20
36084    15
41884    14
41740    12
40900    11
12540     7
23420     5
Name: count, dtype: int64


In [20]:
import numpy as np

print("=== DUPLICATES ===")
print("Duplicate CCNs:", ca_ipps["Provider Number"].duplicated().sum())

print("\n=== CCN FORMAT ===")
bad_ccn = ca_ipps[~ca_ipps["Provider Number"].str.match(r"^05\d{4}$")]
print("CCNs not matching ^05\\d{4}$:", len(bad_ccn))
if len(bad_ccn) > 0:
    print(bad_ccn[["Provider Number", "Name"]])

print("\n=== KEY COLUMNS: missing + dtype + range ===")
key_cols = [
    "FY 2026 Wage Index", "DSHPCT", "DSHOPP", "UCP Per Claim Amount",
    "Resident to Bed Ratio", "Beds", "GAF", "Capital Cost of Living Adjustment",
    "Operating CCR", "Capital CCR", "Payment Labor Market Area",
]
for c in key_cols:
    s = ca_ipps[c]
    print(f"\n{c}")
    print(f"  dtype: {s.dtype}")
    print(f"  missing: {s.isna().sum()}")
    if pd.api.types.is_numeric_dtype(s):
        print(f"  min/median/max: {s.min():.4f} / {s.median():.4f} / {s.max():.4f}")
    else:
        print(f"  unique values (sample): {s.dropna().unique()[:10]}")

print("\n=== PAYMENT CBSA TYPE CHECK ===")
print("Sample raw values:", ca_ipps["Payment Labor Market Area"].head(10).tolist())
print("Are any numeric vs string mixed?", ca_ipps["Payment Labor Market Area"].apply(type).value_counts())

=== DUPLICATES ===
Duplicate CCNs: 0

=== CCN FORMAT ===
CCNs not matching ^05\d{4}$: 0

=== KEY COLUMNS: missing + dtype + range ===

FY 2026 Wage Index
  dtype: float64
  missing: 0
  min/median/max: 1.4315 / 1.4315 / 1.7328

DSHPCT
  dtype: float64
  missing: 0
  min/median/max: 0.0000 / 0.4292 / 1.0595

DSHOPP
  dtype: float64
  missing: 0
  min/median/max: 0.0000 / 0.0500 / 0.1916

UCP Per Claim Amount
  dtype: float64
  missing: 0
  min/median/max: 0.0000 / 594.5700 / 50858.0300

Resident to Bed Ratio
  dtype: float64
  missing: 0
  min/median/max: 0.0000 / 0.0000 / 1.6161

Beds
  dtype: int64
  missing: 0
  min/median/max: 6.0000 / 196.0000 / 908.0000

GAF
  dtype: float64
  missing: 0
  min/median/max: 1.2785 / 1.2785 / 1.4571

Capital Cost of Living Adjustment
  dtype: float64
  missing: 0
  min/median/max: 1.0000 / 1.0000 / 1.0000

Operating CCR
  dtype: float64
  missing: 0
  min/median/max: 0.0650 / 0.2100 / 0.7790

Capital CCR
  dtype: float64
  missing: 0
  min/median/max

In [21]:
print("WI distribution:")
print(ca_ipps["FY 2026 Wage Index"].value_counts().head(10))

print("\nWI for CBSA = 5 (rural statewide):")
print(ca_ipps[ca_ipps["Payment Labor Market Area"] == 5]["FY 2026 Wage Index"].describe())

WI distribution:
FY 2026 Wage Index
1.4315    188
1.4505     12
1.6917      9
1.7197      7
1.6529      6
1.6496      6
1.7020      6
1.7328      6
1.6369      6
1.4850      5
Name: count, dtype: int64

WI for CBSA = 5 (rural statewide):
count    61.000000
mean      1.498289
std       0.112651
min       1.431500
25%       1.431500
50%       1.431500
75%       1.619300
max       1.732800
Name: FY 2026 Wage Index, dtype: float64


In [22]:
print("DSHPCT > 1.0:")
print(ca_ipps[ca_ipps["DSHPCT"] > 1.0][["Provider Number", "Name", "DSHPCT", "DSHOPP"]])

print("\nDSHPCT histogram:")
print(ca_ipps["DSHPCT"].describe(percentiles=[.25,.5,.75,.9,.95,.99]))

DSHPCT > 1.0:
    Provider Number                                               Name  \
254          050126                       Valley Presbyterian Hospital   
268          050149              California Hospital Medical Center La   
280          050192                           Adventist Health Reedley   
343          050378                    Pacifica Hospital Of The Valley   
385          050543                        College Hospital Costa Mesa   
428          050717  Lac/Rancho Los Amigos National Rehabilitation ...   
458          050776                             College Medical Center   

     DSHPCT   DSHOPP  
254  1.0017  0.17964  
268  1.0060  0.18053  
280  1.0321  0.03000  
343  1.0271  0.18488  
385  1.0012  0.17954  
428  1.0230  0.03000  
458  1.0595  0.19156  

DSHPCT histogram:
count    275.000000
mean       0.458225
std        0.238994
min        0.000000
25%        0.278500
50%        0.429200
75%        0.608250
90%        0.794520
95%        0.886570
99%       

In [23]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

# Load Table 3 (Wage Index by CBSA)
xl = pd.ExcelFile(BASE / "CMS-1833-F Tables 2, 3, 4A, 4B.xlsx")
print("Sheets:", xl.sheet_names)

t3 = pd.read_excel(BASE / "CMS-1833-F Tables 2, 3, 4A, 4B.xlsx",
                   sheet_name="Table 3", header=None)
print("\nTable 3 first 5 rows raw:")
print(t3.head())

Sheets: ['Table 2', 'Table 3', 'Table 4A', 'Table 4B']

Table 3 first 5 rows raw:
                                                  0          1      2   \
0  Table 3- WAGE INDEX TABLE BY CBSA  - FY 2026  ...        NaN    NaN   
1                                               CBSA  Area Name  State   
2                                                 01    ALABAMA     AL   
3                                                 02     ALASKA     AK   
4                                                 03    ARIZONA     AZ   

           3                             4   \
0         NaN                           NaN   
1  State Code  2FY 2026 Average Hourly Wage   
2          01                       36.9782   
3          02                       65.9288   
4          03                       49.4162   

                                               5           6       7   \
0                                             NaN         NaN     NaN   
1  23-Year Average Hourly Wage (2024, 2025, 

In [24]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

t3 = pd.read_excel(BASE / "CMS-1833-F Tables 2, 3, 4A, 4B.xlsx",
                   sheet_name="Table 3", header=1, dtype={"CBSA": str, "State Code": str})

print("Columns:", t3.columns.tolist())
print("\nAll California rows:")
ca_rows = t3[t3["State"] == "CA"]
print(ca_rows[["CBSA", "Area Name", "State", "Wage Index", "Reclassified Wage Index"]])

Columns: ['CBSA', 'Area Name', 'State', 'State Code', '2FY 2026 Average Hourly Wage', '23-Year Average Hourly Wage (2024, 2025, 2026)', 'Wage Index', 'GAF', 'Reclassified Wage Index', 'Reclassified GAF', 'State Imputed Floor', 'Eligible for Frontier Wage Index', 'Eligible for Rural Floor Wage Index', 'Eligible for Imputed Floor Wage Index', '3Pre-Frontier and/or Pre-Rural Floor and/or Pre-Imputed Floor Wage Index', 'Reclassified Wage Index Eligible for Frontier Wage Index', 'Reclassified Wage Index Eligible for Rural Floor Wage Index', 'Reclassified Wage Index Eligible for Imputed Floor Wage Index', '3Reclassified Wage Index Pre-Frontier and/or Pre-Rural Floor']

All California rows:
      CBSA                                 Area Name State  Wage Index  \
7       05                                CALIFORNIA    CA      1.4315   
108  11244              Anaheim-Santa Ana-Irvine, CA    CA      1.4315   
127  12540                    Bakersfield-Delano, CA    CA      1.4315   
189  17020 

In [25]:
# 1. Confirm all 275 hospitals have plausible CA names
print("=== Sample of names from your 275 'CA' hospitals ===")
print(ca_ipps[["Provider Number", "Name"]].sample(20, random_state=42).to_string())

# 2. Cross-check with Table 2 — does Table 2 also show these CCNs as CA?
import pandas as pd
from pathlib import Path
BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

t2 = pd.read_excel(BASE / "CMS-1833-F Tables 2, 3, 4A, 4B.xlsx",
                   sheet_name="Table 2", header=1, dtype={"CCN": str})

# What CBSAs do your 275 CCNs map to in Table 2?
your_ccns = set(ca_ipps["Provider Number"])
t2_match = t2[t2["CCN"].isin(your_ccns)]
print(f"\n=== Cross-check ===")
print(f"Your CA hospitals: {len(your_ccns)}")
print(f"Found in Table 2: {len(t2_match)}")
print(f"Missing from Table 2: {len(your_ccns - set(t2['CCN']))}")

# 3. Are all the CBSAs they map to actually CA CBSAs?
ca_cbsas = set(["05","11244","12540","17020","20940","23420","25260","31084",
                "32900","33700","34900","36084","37100","39820","40140","40900",
                "41500","41740","41884","41940","42020","42034","42100","42200",
                "42220","44700","46700","47300","49700"])
t2_match_cbsa = t2_match["Geographic CBSA"].astype(str).str.zfill(5)
non_ca = t2_match[~t2_match_cbsa.isin(ca_cbsas)]
print(f"Hospitals mapped to non-CA CBSAs: {len(non_ca)}")
if len(non_ca) > 0:
    print(non_ca[["CCN", "Geographic CBSA"]].head())

=== Sample of names from your 275 'CA' hospitals ===
    Provider Number                                             Name
221          050069                   Providence St. Joseph Hospital
332          050336                   Adventist Health Lodi Memorial
389          050557                          Memorial Medical Center
317          050295                                   Mercy Hospital
409          050636                     Palomar Medical Center Poway
427          050714  Sutter Maternity & Surgery Center Of Santa Cruz
444          050757                 Alvarado Hospital Medical Center
396          050586                      Chino Valley Medical Center
447          050761   Providence Cedars Sinai Tarzana Medical Center
335          050351                 Torrance Memorial Medical Center
270          050152               Ucsf Health Saint Francis Hospital
391          050567                      Providence Mission Hospital
421          050697                    Patients' H

KeyError: 'CCN'

In [26]:
import pandas as pd
from pathlib import Path
BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

t2 = pd.read_excel(BASE / "CMS-1833-F Tables 2, 3, 4A, 4B.xlsx",
                   sheet_name="Table 2", header=1)

print("Table 2 columns:", t2.columns.tolist())
print("\nFirst 2 rows:")
print(t2.head(2))

Table 2 columns: ['1CCN', '2Case-Mix Indexes for  Discharges Occurring in Federal Fiscal Year 2024', 'FY 2025 Wage Index', '8FY 2025 Wage Index Differs from FY 2025 CN Table 2', '6FY 2026 Wage Index Prior to Cap', '3,6 FY 2026 Wage Index With Cap', '9 FY 2026 Transition for the Discontinuation of the Low Wage Index Hospital Policy', '4Average Hourly Wage FY 2024', '4Average Hourly Wage FY\xa02025', '4Average Hourly Wage FY\xa02026', '43-Year Average Hourly Wage (2024, 2025, 2026)', 'Geographic CBSA', '7Wage Index Payment CBSA', 'Lugar/ NECMA', 'MGCRB Reclass', 'Rural CBSA if Hospital is Reclassified Under Section 1886(d)(8)(E)of the Act (412.103)', '5Out-Migration Adjustment', 'County Name', 'FIPS County Code', 'Dual Status 412.103 and MGCRB/ LUGAR', 'MGCRB Reclass to Home', 'Hospitals that Waive LUGAR to Receive Out-Migration Adjustment']

First 2 rows:
     1CCN  \
0  010001   
1  010005   

   2Case-Mix Indexes for  Discharges Occurring in Federal Fiscal Year 2024  \
0              

In [27]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

# Reload Table 2 with proper string types
t2 = pd.read_excel(BASE / "CMS-1833-F Tables 2, 3, 4A, 4B.xlsx",
                   sheet_name="Table 2", header=1,
                   dtype={"1CCN": str})

# Rename for sanity
t2 = t2.rename(columns={"1CCN": "CCN", "Geographic CBSA": "geo_cbsa"})

# 1. Eyeball test — sample 20 names from your CA universe
print("=== EYEBALL TEST: 20 random hospitals from your 'CA' set ===")
print(ca_ipps[["Provider Number", "Name"]].sample(20, random_state=42).to_string(index=False))

# 2. Round-trip check
your_ccns = set(ca_ipps["Provider Number"])
t2_ccns = set(t2["CCN"].dropna())
print(f"\n=== ROUND-TRIP CHECK ===")
print(f"Your CA hospitals: {len(your_ccns)}")
print(f"Of those, found in Table 2: {len(your_ccns & t2_ccns)}")
print(f"Missing from Table 2: {len(your_ccns - t2_ccns)}")

# 3. CBSA membership check
ca_cbsas = {"5","11244","12540","17020","20940","23420","25260","31084",
            "32900","33700","34900","36084","37100","39820","40140","40900",
            "41500","41740","41884","41940","42020","42034","42100","42200",
            "42220","44700","46700","47300","49700"}

t2_match = t2[t2["CCN"].isin(your_ccns)].copy()
# Convert CBSA to clean string (strip ".0" from float representation)
t2_match["geo_cbsa_str"] = t2_match["geo_cbsa"].apply(
    lambda x: str(int(x)) if pd.notna(x) else None
)

non_ca = t2_match[~t2_match["geo_cbsa_str"].isin(ca_cbsas)]
print(f"\n=== CBSA CHECK ===")
print(f"Hospitals from your set with non-CA Geographic CBSA: {len(non_ca)}")
if len(non_ca) > 0:
    print(non_ca[["CCN", "geo_cbsa_str"]].head(10))
else:
    print("✅ All 275 hospitals are in California CBSAs.")

=== EYEBALL TEST: 20 random hospitals from your 'CA' set ===
Provider Number                                            Name
         050069                  Providence St. Joseph Hospital
         050336                  Adventist Health Lodi Memorial
         050557                         Memorial Medical Center
         050295                                  Mercy Hospital
         050636                    Palomar Medical Center Poway
         050714 Sutter Maternity & Surgery Center Of Santa Cruz
         050757                Alvarado Hospital Medical Center
         050586                     Chino Valley Medical Center
         050761  Providence Cedars Sinai Tarzana Medical Center
         050351                Torrance Memorial Medical Center
         050152              Ucsf Health Saint Francis Hospital
         050567                     Providence Mission Hospital
         050697                   Patients' Hospital Of Redding
         050776                          Co

In [28]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

# Inspect Table 5 structure first
xl5 = pd.ExcelFile(BASE / "CMS-1833-F Table 5.xlsx")
print("Sheets:", xl5.sheet_names)

t5 = pd.read_excel(BASE / "CMS-1833-F Table 5.xlsx",
                   sheet_name=xl5.sheet_names[0],
                   header=1,
                   dtype={"MS-DRG": str})

print("\nColumns:", t5.columns.tolist())
print("\nDRG 470 row:")
print(t5[t5["MS-DRG"] == "470"])

Sheets: ['FY 2026 Table 5 FR']

Columns: ['MS-DRG ', 'FY 2026 Final Post-Acute DRG', 'FY 2026 Final Special Pay DRG', 'MDC', 'TYPE', 'MS-DRG Title', 'Weights - Before Cap', 'Weights - 10% Cap Applied ', 'Geometric mean LOS', 'Arithmetic mean LOS']

DRG 470 row:


KeyError: 'MS-DRG'

In [29]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

xl5 = pd.ExcelFile(BASE / "CMS-1833-F Table 5.xlsx")
print("Sheets:", xl5.sheet_names)

t5 = pd.read_excel(BASE / "CMS-1833-F Table 5.xlsx",
                   sheet_name=xl5.sheet_names[0],
                   header=1)

print("\nColumns:", t5.columns.tolist())
print("\nFirst 3 rows:")
print(t5.head(3))

Sheets: ['FY 2026 Table 5 FR']

Columns: ['MS-DRG ', 'FY 2026 Final Post-Acute DRG', 'FY 2026 Final Special Pay DRG', 'MDC', 'TYPE', 'MS-DRG Title', 'Weights - Before Cap', 'Weights - 10% Cap Applied ', 'Geometric mean LOS', 'Arithmetic mean LOS']

First 3 rows:
   MS-DRG  FY 2026 Final Post-Acute DRG FY 2026 Final Special Pay DRG  MDC  \
0        1                           No                            No  PRE   
1        2                           No                            No  PRE   
2        3                          Yes                            No  PRE   

   TYPE                                       MS-DRG Title  \
0  SURG  HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...   
1  SURG  HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...   
2  SURG  ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...   

  Weights - Before Cap Weights - 10% Cap Applied  Geometric mean LOS  \
0              28.0239                    28.0239               25.8   
1              11.3318         

In [30]:
# Strip whitespace from all column names
t5.columns = t5.columns.str.strip()

# MS-DRG values are integers, not strings — let's check the type
print("MS-DRG dtype:", t5["MS-DRG"].dtype)
print("First values:", t5["MS-DRG"].head().tolist())

# Filter to DRG 470 (try int first since MS-DRG looks numeric)
drg470 = t5[t5["MS-DRG"] == 470]
print("\nDRG 470:")
print(drg470[["MS-DRG", "MS-DRG Title", "Weights - Before Cap", "Weights - 10% Cap Applied", "Geometric mean LOS"]])

MS-DRG dtype: int64
First values: [1, 2, 3, 4, 5]

DRG 470:
     MS-DRG                                       MS-DRG Title  \
382     470  MAJOR HIP AND KNEE JOINT REPLACEMENT OR REATTA...   

    Weights - Before Cap Weights - 10% Cap Applied Geometric mean LOS  
382               1.9289                    1.9289                1.9  


In [31]:
DRG470_WEIGHT = 1.9289
print(f"Using FY2026 DRG 470 weight: {DRG470_WEIGHT}")

Using FY2026 DRG 470 weight: 1.9289


In [32]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")

# Reload the full Impact File (we filtered to CA earlier; now we need TX too)
impact = pd.read_excel(
    BASE / "FY 2026 IPPS Final Rule Impact File.xlsx",
    sheet_name="FY 2026 FR",
    header=1,
    dtype={"Provider Number": str}
)

# === FY 2026 CONSTANTS ===
LABOR_SHARE     = 4456.72   # Table 1A, WI > 1
NONLABOR_SHARE  = 2295.89   # Table 1A, WI > 1
CAPITAL_RATE    = 524.15    # Table 1D
DRG470_WEIGHT   = 1.9289    # Table 5

# === IPPS PAYMENT FORMULA ===
def compute_drg470_payment(row):
    wi   = row["FY 2026 Wage Index"]
    rbr  = row["Resident to Bed Ratio"]
    dshopp = row["DSHOPP"]
    ucp  = row["UCP Per Claim Amount"]
    gaf  = row["GAF"]
    ccola = row["Capital Cost of Living Adjustment"]

    # Operating Base
    wage_adjusted = (LABOR_SHARE * wi) + NONLABOR_SHARE
    op_base = wage_adjusted * DRG470_WEIGHT

    # IME (only for teaching hospitals; rbr = 0 for non-teaching → IME = 0)
    if rbr > 0:
        ime = op_base * 1.35 * ((1 + rbr) ** 0.405 - 1)
    else:
        ime = 0.0

    # DSH (operating)
    dsh = op_base * dshopp

    # UCP (already per-claim)
    ucp_payment = ucp

    # Capital
    capital = CAPITAL_RATE * gaf * DRG470_WEIGHT * ccola

    total = op_base + ime + dsh + ucp_payment + capital

    return pd.Series({
        "Operating Base": round(op_base, 2),
        "IME":            round(ime, 2),
        "DSH":            round(dsh, 2),
        "UCP":            round(ucp_payment, 2),
        "Capital":        round(capital, 2),
        "TOTAL":          round(total, 2),
    })

# === VALIDATION: Baylor (450021) and Parkland (450623) ===
test_ccns = ["450021", "450623"]
test = impact[impact["Provider Number"].isin(test_ccns)].copy()

print("Found:")
print(test[["Provider Number", "Name", "Payment Labor Market Area",
            "FY 2026 Wage Index", "Resident to Bed Ratio",
            "DSHOPP", "UCP Per Claim Amount", "GAF"]])

print("\n=== COMPUTED PAYMENTS ===")
results = test.apply(compute_drg470_payment, axis=1)
results.insert(0, "Hospital", test["Name"].values)
results.insert(0, "CCN", test["Provider Number"].values)
print(results.to_string(index=False))

print("\n=== COTTER'S PUBLISHED NUMBERS ===")
print("Baylor (450021):  $19,395")
print("Parkland (450623): $92,451")

Found:
     Provider Number                              Name  \
2572          450021  Baylor University Medical Center   

      Payment Labor Market Area  FY 2026 Wage Index  Resident to Bed Ratio  \
2572                         45              0.9721                 0.2161   

       DSHOPP  UCP Per Claim Amount       GAF  
2572  0.03984               3536.63  0.980809  

=== COMPUTED PAYMENTS ===
   CCN                         Hospital  Operating Base    IME    DSH     UCP  Capital    TOTAL
450021 Baylor University Medical Center        12785.27 1423.3 509.36 3536.63   991.63 19246.19

=== COTTER'S PUBLISHED NUMBERS ===
Baylor (450021):  $19,395
Parkland (450623): $92,451


In [33]:
# Search for Parkland by name
parkland_search = impact[impact["Name"].str.contains("PARKLAND", case=False, na=False)]
print("Parkland matches by name:")
print(parkland_search[["Provider Number", "Name"]])

# Also check all 450xxx CCNs near 450623
print("\nNearby Texas CCNs (450620-450630):")
nearby = impact[impact["Provider Number"].between("450620", "450630")]
print(nearby[["Provider Number", "Name"]])

Parkland matches by name:
     Provider Number                                 Name
1609          260163               Parkland Health Center
1688          300017              Parkland Medical Center
2570          450015  Parkland Health And Hospital System

Nearby Texas CCNs (450620-450630):
Empty DataFrame
Columns: [Provider Number, Name]
Index: []


In [34]:
# Correct CCNs
test_ccns = ["450021", "450015"]   # Baylor, Parkland
test = impact[impact["Provider Number"].isin(test_ccns)].copy()

print("Found:")
print(test[["Provider Number", "Name", "Payment Labor Market Area",
            "FY 2026 Wage Index", "Resident to Bed Ratio",
            "DSHOPP", "UCP Per Claim Amount", "GAF"]])

print("\n=== COMPUTED PAYMENTS ===")
results = test.apply(compute_drg470_payment, axis=1)
results.insert(0, "Hospital", test["Name"].values)
results.insert(0, "CCN", test["Provider Number"].values)
print(results.to_string(index=False))

print("\n=== COTTER'S PUBLISHED NUMBERS ===")
print("Baylor   (450021): $19,395")
print("Parkland (450015): $92,451")

Found:
     Provider Number                                 Name  \
2570          450015  Parkland Health And Hospital System   
2572          450021     Baylor University Medical Center   

      Payment Labor Market Area  FY 2026 Wage Index  Resident to Bed Ratio  \
2570                         45              0.9721                 0.9301   
2572                         45              0.9721                 0.2161   

       DSHOPP  UCP Per Claim Amount       GAF  
2570  0.15056              71888.02  0.980809  
2572  0.03984               3536.63  0.980809  

=== COMPUTED PAYMENTS ===
   CCN                            Hospital  Operating Base     IME     DSH      UCP  Capital    TOTAL
450015 Parkland Health And Hospital System        12785.27 5266.89 1924.95 71888.02   991.63 92856.76
450021    Baylor University Medical Center        12785.27 1423.30  509.36  3536.63   991.63 19246.19

=== COTTER'S PUBLISHED NUMBERS ===
Baylor   (450021): $19,395
Parkland (450015): $92,451


In [35]:
# Run the same formula across the entire CA universe
ca_payments = ca_ipps.apply(compute_drg470_payment, axis=1)

# Attach hospital identifiers
ca_results = pd.concat([
    ca_ipps[["Provider Number", "Name", "Payment Labor Market Area",
             "FY 2026 Wage Index", "DSHOPP", "Resident to Bed Ratio",
             "UCP Per Claim Amount", "Beds", "Provider Type"]].reset_index(drop=True),
    ca_payments.reset_index(drop=True)
], axis=1)

# Save
ca_results.to_csv(PROC / "ca_drg470_payments.csv", index=False)
print(f"Computed payments for {len(ca_results)} CA hospitals.")
print(f"Saved to: {PROC / 'ca_drg470_payments.csv'}\n")

# === DESCRIPTIVE STATISTICS ===
print("=== DRG 470 TOTAL PAYMENT — STATEWIDE DISTRIBUTION ===")
print(ca_results["TOTAL"].describe(percentiles=[.05,.10,.25,.5,.75,.90,.95,.99]).round(2))

print(f"\nMin / Max ratio statewide: {ca_results['TOTAL'].max() / ca_results['TOTAL'].min():.2f}x")

print("\n=== TOP 5 HIGHEST-PAID HOSPITALS ===")
print(ca_results.nlargest(5, "TOTAL")[["Provider Number","Name","Payment Labor Market Area","TOTAL","UCP","IME","DSH"]].to_string(index=False))

print("\n=== BOTTOM 5 LOWEST-PAID HOSPITALS ===")
print(ca_results.nsmallest(5, "TOTAL")[["Provider Number","Name","Payment Labor Market Area","TOTAL","UCP","IME","DSH"]].to_string(index=False))

Computed payments for 275 CA hospitals.
Saved to: C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data\processed\ca_drg470_payments.csv

=== DRG 470 TOTAL PAYMENT — STATEWIDE DISTRIBUTION ===
count      275.00
mean     21868.54
std       5383.79
min      18027.09
5%       18529.13
10%      18701.94
25%      19589.48
50%      20671.96
75%      22171.80
90%      25304.20
95%      28250.46
99%      39864.83
max      71942.55
Name: TOTAL, dtype: float64

Min / Max ratio statewide: 3.99x

=== TOP 5 HIGHEST-PAID HOSPITALS ===
Provider Number                                               Name  Payment Labor Market Area    TOTAL      UCP     IME     DSH
         050717 Lac/Rancho Los Amigos National Rehabilitation  Ctr                      31084 71942.55 50858.03 2555.39  502.04
         050778          Loma Linda University Children's Hospital                      40140 68966.29 48030.07    0.00 2909.13
         050192                           Adventist Health Reedley            

In [37]:
print("Provider Type breakdown for top 10:")
print(ca_results.nlargest(10, "TOTAL")[
    ["Provider Number","Name","Provider Type","Beds","TOTAL"]
].to_string(index=False))

print("\nProvider Type legend:")
print("  0  = Short-term acute care hospital")
print("  7  = Rural Referral Center")
print("  16 = Medicare-Dependent Hospital")

print("\nThe $18,027 floor group:")
floor = ca_results[ca_results["TOTAL"] < 18030]
print(f"Count: {len(floor)}")
print(floor[
    ["Provider Number","Name","Provider Type","Beds","Resident to Bed Ratio","DSHOPP"]
].to_string(index=False))

Provider Type breakdown for top 10:
Provider Number                                               Name  Provider Type  Beds    TOTAL
         050717 Lac/Rancho Los Amigos National Rehabilitation  Ctr              0    58 71942.55
         050778          Loma Linda University Children's Hospital              0   364 68966.29
         050192                           Adventist Health Reedley              0    49 42768.71
         050040                 Lac/Olive View-Ucla Medical Center              0   241 38844.55
         050373                 Los Angeles General Medical Center              0   564 35552.24
         050376                         Lac/Harbor-Ucla Med Center              0   374 35527.48
         050276               Contra Costa Regional Medical Center              0   124 35505.85
         050228 Zuckerberg San Francisco General Hosp & Trauma Ctr              0   284 35431.96
         050320                                  Highland Hospital              0   232 345

In [38]:
import pandas as pd

# Exclusion list with reasoning (for methods writeup)
exclusions = pd.DataFrame([
    {"CCN": "050717", "Name": "LAC/Rancho Los Amigos",
     "Reason": "Specialty rehabilitation hospital; minimal DRG 470 volume"},
    {"CCN": "050778", "Name": "Loma Linda University Children's Hospital",
     "Reason": "Children's hospital; adult hip/knee replacement not core service"},
    {"CCN": "050192", "Name": "Adventist Health Reedley",
     "Reason": "Small rural hospital (49 beds); per-claim UCP of $22,914 indicates volume artifact"},
])
print("=== EXCLUSIONS ===")
print(exclusions.to_string(index=False))

excluded_ccns = set(exclusions["CCN"])
ca_clean = ca_results[~ca_results["Provider Number"].isin(excluded_ccns)].copy()

print(f"\nBefore exclusions: {len(ca_results)}")
print(f"After exclusions:  {len(ca_clean)}")

# Save
ca_clean.to_csv(PROC / "ca_drg470_payments_clean.csv", index=False)
print(f"Saved: {PROC / 'ca_drg470_payments_clean.csv'}")

# === DISTRIBUTION ===
print("\n=== DISTRIBUTION AFTER EXCLUSIONS ===")
print(ca_clean["TOTAL"].describe(percentiles=[.05,.10,.25,.5,.75,.90,.95,.99]).round(2))

ratio_statewide = ca_clean["TOTAL"].max() / ca_clean["TOTAL"].min()
print(f"\nStatewide max/min ratio: {ratio_statewide:.2f}x")

# Top 5 and bottom 5 after cleaning
print("\n=== TOP 5 AFTER EXCLUSIONS ===")
print(ca_clean.nlargest(5, "TOTAL")[
    ["Provider Number","Name","Payment Labor Market Area","TOTAL","UCP","IME","DSH"]
].to_string(index=False))

print("\n=== BOTTOM 5 AFTER EXCLUSIONS ===")
print(ca_clean.nsmallest(5, "TOTAL")[
    ["Provider Number","Name","Payment Labor Market Area","TOTAL","UCP","IME","DSH"]
].to_string(index=False))

=== EXCLUSIONS ===
   CCN                                      Name                                                                             Reason
050717                     LAC/Rancho Los Amigos                          Specialty rehabilitation hospital; minimal DRG 470 volume
050778 Loma Linda University Children's Hospital                   Children's hospital; adult hip/knee replacement not core service
050192                  Adventist Health Reedley Small rural hospital (49 beds); per-claim UCP of $22,914 indicates volume artifact

Before exclusions: 275
After exclusions:  272
Saved: C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data\processed\ca_drg470_payments_clean.csv

=== DISTRIBUTION AFTER EXCLUSIONS ===
count      272.00
mean     21434.45
std       3172.92
min      18027.09
5%       18529.13
10%      18701.02
25%      19579.49
50%      20659.02
75%      22121.96
90%      25112.84
95%      26937.18
99%      35512.12
max      38844.55
Name: TOTAL, dtype: fl

In [39]:
from itertools import combinations

# For each CBSA with 2+ hospitals, form all possible pairs
pairs = []
for cbsa, grp in ca_clean.groupby("Payment Labor Market Area"):
    if len(grp) < 2:
        continue
    rows = grp.to_dict("records")
    for a, b in combinations(rows, 2):
        # Only keep pairs with same wage index (within 0.01 tolerance)
        if abs(a["FY 2026 Wage Index"] - b["FY 2026 Wage Index"]) > 0.01:
            continue
        lo, hi = sorted([a["TOTAL"], b["TOTAL"]])
        pairs.append({
            "cbsa": cbsa,
            "ccn_a": a["Provider Number"], "name_a": a["Name"], "pay_a": a["TOTAL"],
            "ccn_b": b["Provider Number"], "name_b": b["Name"], "pay_b": b["TOTAL"],
            "gap_abs":  hi - lo,
            "gap_ratio": hi / lo,
            "wi_same": abs(a["FY 2026 Wage Index"] - b["FY 2026 Wage Index"]) < 0.0001,
        })

pairs_df = pd.DataFrame(pairs)
pairs_df.to_csv(PROC / "ca_pair_ratios.csv", index=False)

print(f"Total valid pairs (same CBSA + same wage index): {len(pairs_df)}")
print(f"\n=== DISTRIBUTION OF PAYMENT GAP RATIOS ===")
print(pairs_df["gap_ratio"].describe(percentiles=[.05,.25,.5,.75,.90,.95,.99]).round(3))

# How common is a Cotter-sized gap?
for threshold in [1.5, 2.0, 3.0, 4.0, 4.77]:
    pct = (pairs_df["gap_ratio"] >= threshold).mean() * 100
    n = (pairs_df["gap_ratio"] >= threshold).sum()
    print(f"Pairs with ratio >= {threshold}x: {n:>5,} ({pct:.1f}%)")

print("\n=== TOP 10 BIGGEST GAP PAIRS ===")
top = pairs_df.nlargest(10, "gap_ratio")
print(top[["cbsa","name_a","pay_a","name_b","pay_b","gap_ratio"]].to_string(index=False))

Total valid pairs (same CBSA + same wage index): 2927

=== DISTRIBUTION OF PAYMENT GAP RATIOS ===
count    2927.000
mean        1.144
std         0.191
min         1.000
5%          1.006
25%         1.036
50%         1.082
75%         1.154
90%         1.333
95%         1.639
99%         1.914
max         2.155
Name: gap_ratio, dtype: float64
Pairs with ratio >= 1.5x:   191 (6.5%)
Pairs with ratio >= 2.0x:    13 (0.4%)
Pairs with ratio >= 3.0x:     0 (0.0%)
Pairs with ratio >= 4.0x:     0 (0.0%)
Pairs with ratio >= 4.77x:     0 (0.0%)

=== TOP 10 BIGGEST GAP PAIRS ===
 cbsa                             name_a    pay_a                                 name_b    pay_b  gap_ratio
31084 Lac/Olive View-Ucla Medical Center 38844.55                 Docs Surgical Hospital 18027.09   2.154788
31084 Lac/Olive View-Ucla Medical Center 38844.55 Kaiser Foundation Hospital - South Bay 18175.51   2.137192
31084 Lac/Olive View-Ucla Medical Center 38844.55             Saint John's Health Center 18560.11

In [40]:
import pandas as pd
from itertools import combinations

# Rebuild pairs with component-level detail
pairs_detailed = []
for cbsa, grp in ca_clean.groupby("Payment Labor Market Area"):
    if len(grp) < 2:
        continue
    rows = grp.to_dict("records")
    for a, b in combinations(rows, 2):
        if abs(a["FY 2026 Wage Index"] - b["FY 2026 Wage Index"]) > 0.01:
            continue
        # Order so "hi" = higher-paid, "lo" = lower-paid
        if a["TOTAL"] >= b["TOTAL"]:
            hi, lo = a, b
        else:
            hi, lo = b, a
        pairs_detailed.append({
            "cbsa": cbsa,
            "name_hi": hi["Name"], "name_lo": lo["Name"],
            "pay_hi":  hi["TOTAL"], "pay_lo": lo["TOTAL"],
            "gap_ratio": hi["TOTAL"] / lo["TOTAL"],
            "gap_abs":   hi["TOTAL"] - lo["TOTAL"],
            "gap_base":  hi["Operating Base"] - lo["Operating Base"],
            "gap_ime":   hi["IME"]     - lo["IME"],
            "gap_dsh":   hi["DSH"]     - lo["DSH"],
            "gap_ucp":   hi["UCP"]     - lo["UCP"],
            "gap_cap":   hi["Capital"] - lo["Capital"],
        })

pd_df = pd.DataFrame(pairs_detailed)

# Compute what % of each gap comes from each component
for c in ["base","ime","dsh","ucp","cap"]:
    pd_df[f"pct_{c}"] = (pd_df[f"gap_{c}"] / pd_df["gap_abs"] * 100).round(1)

pd_df.to_csv(PROC / "ca_pair_decomposition.csv", index=False)

# === AVERAGE DECOMPOSITION ACROSS ALL PAIRS ===
print("=== AVERAGE SHARE OF GAP BY COMPONENT (all 2,927 pairs) ===")
for c, label in [("base","Operating Base"), ("ime","IME"), ("dsh","DSH"),
                  ("ucp","UCP"), ("cap","Capital")]:
    mean_pct = pd_df[f"pct_{c}"].mean()
    print(f"  {label:16s}: {mean_pct:6.1f}%")

# === DECOMPOSITION FOR HIGH-GAP PAIRS (>= 1.5x) ===
big = pd_df[pd_df["gap_ratio"] >= 1.5]
print(f"\n=== AVERAGE SHARE OF GAP — HIGH-GAP PAIRS ONLY (ratio >= 1.5x, n={len(big)}) ===")
for c, label in [("base","Operating Base"), ("ime","IME"), ("dsh","DSH"),
                  ("ucp","UCP"), ("cap","Capital")]:
    mean_pct = big[f"pct_{c}"].mean()
    print(f"  {label:16s}: {mean_pct:6.1f}%")

# === TOP 5 HIGH-GAP PAIRS WITH FULL DECOMPOSITION ===
print("\n=== TOP 5 PAIRS — COMPONENT BREAKDOWN ===")
top5 = pd_df.nlargest(5, "gap_ratio")
for _, r in top5.iterrows():
    print(f"\n{r['name_hi']}  vs.  {r['name_lo']}")
    print(f"  Ratio: {r['gap_ratio']:.2f}x   Absolute gap: ${r['gap_abs']:,.0f}")
    print(f"    UCP     contributes ${r['gap_ucp']:>8,.0f}  ({r['pct_ucp']:>5.1f}%)")
    print(f"    IME     contributes ${r['gap_ime']:>8,.0f}  ({r['pct_ime']:>5.1f}%)")
    print(f"    DSH     contributes ${r['gap_dsh']:>8,.0f}  ({r['pct_dsh']:>5.1f}%)")
    print(f"    Capital contributes ${r['gap_cap']:>8,.0f}  ({r['pct_cap']:>5.1f}%)")
    print(f"    Base    contributes ${r['gap_base']:>8,.0f}  ({r['pct_base']:>5.1f}%)")

=== AVERAGE SHARE OF GAP BY COMPONENT (all 2,927 pairs) ===
  Operating Base  :    0.1%
  IME             :  904.0%
  DSH             : -398.3%
  UCP             : -405.8%
  Capital         :    0.0%

=== AVERAGE SHARE OF GAP — HIGH-GAP PAIRS ONLY (ratio >= 1.5x, n=191) ===
  Operating Base  :   -0.0%
  IME             :   41.7%
  DSH             :    8.1%
  UCP             :   50.2%
  Capital         :    0.0%

=== TOP 5 PAIRS — COMPONENT BREAKDOWN ===

Lac/Olive View-Ucla Medical Center  vs.  Docs Surgical Hospital
  Ratio: 2.15x   Absolute gap: $20,817
    UCP     contributes $  13,305  ( 63.9%)
    IME     contributes $   4,973  ( 23.9%)
    DSH     contributes $   2,539  ( 12.2%)
    Capital contributes $       0  (  0.0%)
    Base    contributes $       0  (  0.0%)

Lac/Olive View-Ucla Medical Center  vs.  Kaiser Foundation Hospital - South Bay
  Ratio: 2.14x   Absolute gap: $20,669
    UCP     contributes $  13,305  ( 64.4%)
    IME     contributes $   4,825  ( 23.3%)
    DSH   

In [41]:
# Dollar-weighted decomposition (robust to small-gap pairs)
print("=== DOLLAR-WEIGHTED SHARES (correct version) ===")
total_gap = pd_df["gap_abs"].sum()
for c, label in [("base","Operating Base"), ("ime","IME"), ("dsh","DSH"),
                  ("ucp","UCP"), ("cap","Capital")]:
    share = pd_df[f"gap_{c}"].sum() / total_gap * 100
    print(f"  {label:16s}: {share:6.1f}%")

print(f"\n=== DOLLAR-WEIGHTED — HIGH-GAP PAIRS ONLY (>=1.5x) ===")
total_gap_big = big["gap_abs"].sum()
for c, label in [("base","Operating Base"), ("ime","IME"), ("dsh","DSH"),
                  ("ucp","UCP"), ("cap","Capital")]:
    share = big[f"gap_{c}"].sum() / total_gap_big * 100
    print(f"  {label:16s}: {share:6.1f}%")

=== DOLLAR-WEIGHTED SHARES (correct version) ===
  Operating Base  :   -0.0%
  IME             :   39.3%
  DSH             :   25.3%
  UCP             :   35.5%
  Capital         :   -0.0%

=== DOLLAR-WEIGHTED — HIGH-GAP PAIRS ONLY (>=1.5x) ===
  Operating Base  :   -0.0%
  IME             :   39.8%
  DSH             :    8.0%
  UCP             :   52.2%
  Capital         :   -0.0%


In [42]:
# How many unique "high-paid" hospitals are in the high-gap pairs?
hi_hospitals_in_big_pairs = big["name_hi"].value_counts()
print("=== HOSPITALS DRIVING THE HIGH-GAP PAIRS ===")
print(hi_hospitals_in_big_pairs.head(10))
print(f"\nTotal unique 'high' hospitals in 191 high-gap pairs: {hi_hospitals_in_big_pairs.nunique()}")

=== HOSPITALS DRIVING THE HIGH-GAP PAIRS ===
name_hi
Lac/Olive View-Ucla Medical Center                    50
Los Angeles General Medical Center                    49
Lac/Harbor-Ucla Med Center                            49
Ronald Reagan Ucla Medical Center                     14
Arrowhead Regional Medical Center                      8
Highland Hospital                                      6
Contra Costa Regional Medical Center                   5
Uci Health-Orange                                      5
Zuckerberg San Francisco General Hosp & Trauma Ctr     3
Kern Medical Center                                    2
Name: count, dtype: int64

Total unique 'high' hospitals in 191 high-gap pairs: 8


In [44]:
import pandas as pd
from itertools import combinations
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")
PROC = BASE / "processed"

# Load the clean payments file from earlier
ca_clean = pd.read_csv(PROC / "ca_drg470_payments_clean.csv",
                      dtype={"Provider Number": str})

# === Step 1: Identify rural-floor-bound hospitals ===
# Any hospital with wage index exactly 1.4315 is on the floor
RURAL_FLOOR = 1.4315
on_floor = ca_clean["FY 2026 Wage Index"] == RURAL_FLOOR
ca_organic = ca_clean[~on_floor].copy()

print("=== SAMPLE COMPARISON ===")
print(f"Original (all CA IPPS hospitals): {len(ca_clean)}")
print(f"On rural floor (excluded):        {on_floor.sum()}")
print(f"Organic labor markets only:       {len(ca_organic)}")

print("\n=== CBSAs in the organic-only sample ===")
print(ca_organic["Payment Labor Market Area"].value_counts())

# === Step 2: Generate same-CBSA, same-wage-index pairs (organic only) ===
pairs = []
for cbsa, grp in ca_organic.groupby("Payment Labor Market Area"):
    if len(grp) < 2:
        continue
    rows = grp.to_dict("records")
    for a, b in combinations(rows, 2):
        if abs(a["FY 2026 Wage Index"] - b["FY 2026 Wage Index"]) > 0.01:
            continue
        if a["TOTAL"] >= b["TOTAL"]:
            hi, lo = a, b
        else:
            hi, lo = b, a
        pairs.append({
            "cbsa": cbsa,
            "name_hi": hi["Name"], "pay_hi": hi["TOTAL"],
            "name_lo": lo["Name"], "pay_lo": lo["TOTAL"],
            "gap_ratio": hi["TOTAL"] / lo["TOTAL"],
            "gap_abs":   hi["TOTAL"] - lo["TOTAL"],
            "gap_ucp":   hi["UCP"] - lo["UCP"],
            "gap_ime":   hi["IME"] - lo["IME"],
            "gap_dsh":   hi["DSH"] - lo["DSH"],
        })

org_df = pd.DataFrame(pairs)

print(f"\n=== ORGANIC-ONLY PAIR ANALYSIS ===")
print(f"Total pairs: {len(org_df)}")
print(f"\nGap ratio distribution:")
print(org_df["gap_ratio"].describe(percentiles=[.5,.75,.9,.95,.99]).round(3))

for threshold in [1.5, 2.0, 3.0, 4.0]:
    n = (org_df["gap_ratio"] >= threshold).sum()
    pct = n / len(org_df) * 100
    print(f"Pairs with ratio >= {threshold}x: {n} ({pct:.1f}%)")

# === Step 3: Decompose the gaps ===
print(f"\n=== DOLLAR-WEIGHTED GAP DECOMPOSITION (all organic pairs) ===")
total_gap = org_df["gap_abs"].sum()
for c, label in [("ucp","UCP"), ("ime","IME"), ("dsh","DSH")]:
    share = org_df[f"gap_{c}"].sum() / total_gap * 100
    print(f"  {label}: {share:.1f}%")

big = org_df[org_df["gap_ratio"] >= 1.5]
if len(big) > 0:
    print(f"\n=== HIGH-GAP PAIRS (>= 1.5x), n={len(big)} ===")
    total_gap_big = big["gap_abs"].sum()
    for c, label in [("ucp","UCP"), ("ime","IME"), ("dsh","DSH")]:
        share = big[f"gap_{c}"].sum() / total_gap_big * 100
        print(f"  {label}: {share:.1f}%")

# === Step 4: Side-by-side comparison ===
print(f"\n=== ORIGINAL vs SENSITIVITY (side-by-side) ===")
print(f"{'Metric':<35s} {'Original (all)':<20s} {'Organic only':<20s}")
print("-" * 75)
print(f"{'Hospitals':<35s} {'272':<20s} {len(ca_organic):<20d}")
print(f"{'Pairs':<35s} {'2,927':<20s} {len(org_df):<20d}")
print(f"{'Median gap ratio':<35s} {'1.08x':<20s} {f'{org_df[\"gap_ratio\"].median():.2f}x':<20s}")
print(f"{'Max gap ratio':<35s} {'2.15x':<20s} {f'{org_df[\"gap_ratio\"].max():.2f}x':<20s}")
print(f"{'Pairs >= 2x':<35s} {'13 (0.4%)':<20s} {f'{(org_df[\"gap_ratio\"]>=2).sum()} ({(org_df[\"gap_ratio\"]>=2).mean()*100:.1f}%)':<20s}")

# Save
org_df.to_csv(PROC / "ca_pair_decomposition_organic.csv", index=False)
print(f"\nSaved: {PROC / 'ca_pair_decomposition_organic.csv'}")

SyntaxError: unexpected character after line continuation character (2821250198.py, line 83)

In [45]:
import pandas as pd
from itertools import combinations
from pathlib import Path

BASE = Path(r"C:\Users\deeksha\OneDrive - Indiana University\ipps_project\data")
PROC = BASE / "processed"

# Load the clean payments file from earlier
ca_clean = pd.read_csv(PROC / "ca_drg470_payments_clean.csv",
                       dtype={"Provider Number": str})

# === Step 1: Identify rural-floor-bound hospitals ===
RURAL_FLOOR = 1.4315
on_floor = ca_clean["FY 2026 Wage Index"] == RURAL_FLOOR
ca_organic = ca_clean[~on_floor].copy()

print("=== SAMPLE COMPARISON ===")
print(f"Original (all CA IPPS hospitals): {len(ca_clean)}")
print(f"On rural floor (excluded):        {on_floor.sum()}")
print(f"Organic labor markets only:       {len(ca_organic)}")

print("\n=== CBSAs in the organic-only sample ===")
print(ca_organic["Payment Labor Market Area"].value_counts())

# === Step 2: Generate same-CBSA, same-wage-index pairs (organic only) ===
pairs = []
for cbsa, grp in ca_organic.groupby("Payment Labor Market Area"):
    if len(grp) < 2:
        continue
    rows = grp.to_dict("records")
    for a, b in combinations(rows, 2):
        if abs(a["FY 2026 Wage Index"] - b["FY 2026 Wage Index"]) > 0.01:
            continue
        if a["TOTAL"] >= b["TOTAL"]:
            hi, lo = a, b
        else:
            hi, lo = b, a
        pairs.append({
            "cbsa": cbsa,
            "name_hi": hi["Name"], "pay_hi": hi["TOTAL"],
            "name_lo": lo["Name"], "pay_lo": lo["TOTAL"],
            "gap_ratio": hi["TOTAL"] / lo["TOTAL"],
            "gap_abs":   hi["TOTAL"] - lo["TOTAL"],
            "gap_ucp":   hi["UCP"] - lo["UCP"],
            "gap_ime":   hi["IME"] - lo["IME"],
            "gap_dsh":   hi["DSH"] - lo["DSH"],
        })

org_df = pd.DataFrame(pairs)

print(f"\n=== ORGANIC-ONLY PAIR ANALYSIS ===")
print(f"Total pairs: {len(org_df)}")

if len(org_df) > 0:
    print(f"\nGap ratio distribution:")
    print(org_df["gap_ratio"].describe(percentiles=[.5, .75, .9, .95, .99]).round(3))

    for threshold in [1.5, 2.0, 3.0, 4.0]:
        n = (org_df["gap_ratio"] >= threshold).sum()
        pct = n / len(org_df) * 100
        print(f"Pairs with ratio >= {threshold}x: {n} ({pct:.1f}%)")

    # === Step 3: Decompose the gaps ===
    print(f"\n=== DOLLAR-WEIGHTED GAP DECOMPOSITION (all organic pairs) ===")
    total_gap = org_df["gap_abs"].sum()
    for c, label in [("ucp", "UCP"), ("ime", "IME"), ("dsh", "DSH")]:
        share = org_df[f"gap_{c}"].sum() / total_gap * 100
        print(f"  {label}: {share:.1f}%")

    big = org_df[org_df["gap_ratio"] >= 1.5]
    if len(big) > 0:
        print(f"\n=== HIGH-GAP PAIRS (>= 1.5x), n={len(big)} ===")
        total_gap_big = big["gap_abs"].sum()
        for c, label in [("ucp", "UCP"), ("ime", "IME"), ("dsh", "DSH")]:
            share = big[f"gap_{c}"].sum() / total_gap_big * 100
            print(f"  {label}: {share:.1f}%")
    else:
        print("\nNo pairs with ratio >= 1.5x in the organic-only sample.")

    # === Step 4: Side-by-side comparison ===
    median_org = org_df["gap_ratio"].median()
    max_org = org_df["gap_ratio"].max()
    n_2x = (org_df["gap_ratio"] >= 2).sum()
    pct_2x = (org_df["gap_ratio"] >= 2).mean() * 100

    print(f"\n=== ORIGINAL vs SENSITIVITY (side-by-side) ===")
    print(f"{'Metric':<35s} {'Original (all)':<20s} {'Organic only':<20s}")
    print("-" * 75)
    print(f"{'Hospitals':<35s} {'272':<20s} {len(ca_organic):<20d}")
    print(f"{'Pairs':<35s} {'2,927':<20s} {len(org_df):<20d}")
    print(f"{'Median gap ratio':<35s} {'1.08x':<20s} {median_org:.2f}x")
    print(f"{'Max gap ratio':<35s} {'2.15x':<20s} {max_org:.2f}x")
    print(f"{'Pairs >= 2x':<35s} {'13 (0.4%)':<20s} {n_2x} ({pct_2x:.1f}%)")

    # Save
    org_df.to_csv(PROC / "ca_pair_decomposition_organic.csv", index=False)
    print(f"\nSaved: {PROC / 'ca_pair_decomposition_organic.csv'}")
else:
    print("\nNo valid pairs in the organic-only sample.")

=== SAMPLE COMPARISON ===
Original (all CA IPPS hospitals): 272
On rural floor (excluded):        185
Organic labor markets only:       87

=== CBSAs in the organic-only sample ===
Payment Labor Market Area
5        20
36084    15
41884    14
40900    11
41940     5
44700     5
46700     4
42220     4
42034     3
42100     3
34900     1
41500     1
49700     1
Name: count, dtype: int64

=== ORGANIC-ONLY PAIR ANALYSIS ===
Total pairs: 203

Gap ratio distribution:
count    203.000
mean       1.132
std        0.167
min        1.000
50%        1.063
75%        1.152
90%        1.380
95%        1.539
99%        1.705
max        1.724
Name: gap_ratio, dtype: float64
Pairs with ratio >= 1.5x: 14 (6.9%)
Pairs with ratio >= 2.0x: 0 (0.0%)
Pairs with ratio >= 3.0x: 0 (0.0%)
Pairs with ratio >= 4.0x: 0 (0.0%)

=== DOLLAR-WEIGHTED GAP DECOMPOSITION (all organic pairs) ===
  UCP: 49.8%
  IME: 33.6%
  DSH: 16.5%

=== HIGH-GAP PAIRS (>= 1.5x), n=14 ===
  UCP: 58.3%
  IME: 28.7%
  DSH: 13.0%

=== ORIG